<a href="https://colab.research.google.com/github/eyacharfeddine/alpr-using-domain-adaptation-/blob/main/domain_adaptation%2Bsemi_sup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip  install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
!ls /content/drive/My\ Drive/pguard/thermal/val/images/*.png | wc -l  # Should be 447
!ls /content/drive/My\ Drive/pguard/thermal/val/labels/*.txt | wc -l  # Should be 365 (mismatch!)
!ls /content/drive/My\ Drive/pguard/thermal/train/images/*.png | wc -l  # Should be ~1344 (1441 - 97)

447
546
1341


In [ ]:
!ls /content/drive/My\ Drive/pguard/thermal/val/images/*.png | sort | tail -n 97 | xargs -I {} mv {} /content/drive/My\ Drive/pguard/thermal/train/images/

In [ ]:
%%bash
# Move labels corresponding to the 97 images
ls /content/drive/My\ Drive/pguard/thermal/val/images/*.png | sort | tail -n 97 | xargs -n 1 basename | sed 's/\.png$//' | while read -r file; do
    label_file="/content/drive/My Drive/pguard/thermal/val/labels/${file}.txt"
    if [ -f "$label_file" ]; then
        mv "$label_file" "/content/drive/My Drive/pguard/thermal/train/labels/"
    fi
done

# Check counts
echo "Validation images: $(ls /content/drive/My\ Drive/pguard/thermal/val/images/*.png | wc -l)"  # Should be 350
echo "Validation labels: $(ls /content/drive/My\ Drive/pguard/thermal/val/labels/*.txt | wc -l)"  # Should be ~449 (546 - 97)
echo "Training images: $(ls /content/drive/My\ Drive/pguard/thermal/train/images/*.png | wc -l)"  # Should be 1438

Validation images: 350
Validation labels: 211
Training images: 1433


In [ ]:
%%bash
# Find labels without corresponding images (with proper quoting)
comm -23 \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/val/images/"*.png | xargs -n 1 basename | sed 's/\.png$//' | sort) \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/val/labels/"*.txt | xargs -n 1 basename | sed 's/\.txt$//' | sort) | while IFS= read -r file; do
    label_file="/content/drive/My Drive/pguard/thermal/val/labels/${file}.txt"
    if [ -f "$label_file" ]; then
        mv "$label_file" "/content/drive/My Drive/pguard/thermal/train/labels/"
        echo "Moved extra label $file back to training"
    fi
done

# Verify counts
echo "Validation images: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/images/"*.png | wc -l)"  # Should be 350
echo "Validation labels: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/labels/"*.txt | wc -l)"  # Should be 211
echo "Training images: $(ls -1 "/content/drive/My Drive/pguard/thermal/train/images/"*.png | wc -l)"  # Should be 1433
echo "Training labels: $(ls -1 "/content/drive/My Drive/pguard/thermal/train/labels/"*.txt | wc -l)"  # Should be 1842

Validation images: 350
Validation labels: 211
Training images: 1433
Training labels: 1564


In [ ]:
%%bash
# Re-run the move of extra labels with proper quoting
comm -23 \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/val/images/"*.png | xargs -n 1 basename | sed 's/\.png$//' | sort) \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/val/labels/"*.txt | xargs -n 1 basename | sed 's/\.txt$//' | sort) | while IFS= read -r file; do
    label_file="/content/drive/My Drive/pguard/thermal/val/labels/${file}.txt"
    if [ -f "$label_file" ]; then
        mv "$label_file" "/content/drive/My Drive/pguard/thermal/train/labels/"
        echo "Moved extra label $file back to training"
    fi
done

# Verify counts
echo "Validation images: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/images/"*.png | wc -l)"  # Should be 350
echo "Validation labels: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/labels/"*.txt | wc -l)"  # Should be 211
echo "Training images: $(ls -1 "/content/drive/My Drive/pguard/thermal/train/images/"*.png | wc -l)"  # Should be 1433
echo "Training labels: $(ls -1 "/content/drive/My Drive/pguard/thermal/train/labels/"*.txt | wc -l)"  # Should be 1842 (1564 + 278)

Validation images: 350
Validation labels: 211
Training images: 1433
Training labels: 1564


In [ ]:
!ls -1 "/content/drive/My Drive/pguard/thermal/train/labels/"*.txt | wc -l

1564


In [ ]:
%%bash
# Move labels without corresponding images to training
comm -23 \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/val/images/"*.png | xargs -n 1 basename | sed 's/\.png$//' | sort) \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/val/labels/"*.txt | xargs -n 1 basename | sed 's/\.txt$//' | sort) | while IFS= read -r file; do
    label_file="/content/drive/My Drive/pguard/thermal/val/labels/${file}.txt"
    if [ -f "$label_file" ]; then
        mv "$label_file" "/content/drive/My Drive/pguard/thermal/train/labels/"
        echo "Moved extra label $file back to training"
    fi
done

# Verify counts
echo "Validation images: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/images/"*.png | wc -l)"  # Should be 350
echo "Validation labels: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/labels/"*.txt | wc -l)"  # Should be 211
echo "Training images: $(ls -1 "/content/drive/My Drive/pguard/thermal/train/images/"*.png | wc -l)"  # Should be 1433
echo "Training labels: $(ls -1 "/content/drive/My Drive/pguard/thermal/train/labels/"*.txt | wc -l)"  # Should be 1842 (1564 + 278)

Validation images: 526
Validation labels: 526
Training images: 1256
Training labels: 1248


In [ ]:
!find "/content/drive/My Drive/pguard/thermal/" -type f -name "*.txt" | grep -E "TH_1056|TH_1072|TH_1081" | head -n 3

/content/drive/My Drive/pguard/thermal/train/labels/TH_1056.txt
/content/drive/My Drive/pguard/thermal/train/labels/TH_1081.txt
/content/drive/My Drive/pguard/thermal/train/labels/TH_1072.txt


In [ ]:
%%bash
# Manually count training labels
label_count=$(ls -1 "/content/drive/My Drive/pguard/thermal/train/labels/"*.txt 2>/dev/null | wc -l)
echo "Training labels: $label_count"

# List a sample of labels to confirm
ls -1 "/content/drive/My Drive/pguard/thermal/train/labels/"*.txt | head -n 5

Training labels: 1564
/content/drive/My Drive/pguard/thermal/train/labels/TH_1000.txt
/content/drive/My Drive/pguard/thermal/train/labels/TH_1002.txt
/content/drive/My Drive/pguard/thermal/train/labels/TH_1003.txt
/content/drive/My Drive/pguard/thermal/train/labels/TH_1004.txt
/content/drive/My Drive/pguard/thermal/train/labels/TH_1006.txt


In [ ]:
%%bash
for label in TH_1056 TH_1072 TH_1081; do
    if [ -f "/content/drive/My Drive/pguard/thermal/train/labels/${label}.txt" ]; then
        echo "Found $label.txt"
    else
        echo "$label.txt not found"
    fi
done

Found TH_1056.txt
Found TH_1072.txt
Found TH_1081.txt


In [ ]:
%%bash
# Find pairs of images and labels in training
comm -12 \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/train/images/"*.png | xargs -n 1 basename | sed 's/\.png$//' | sort) \
  <(ls -1 "/content/drive/My Drive/pguard/thermal/train/labels/"*.txt | xargs -n 1 basename | sed 's/\.txt$//' | sort) | shuf -n 139 > /tmp/selected_files.txt

# Move paired files to validation
while IFS= read -r file; do
    img_file="/content/drive/My Drive/pguard/thermal/train/images/${file}.png"
    label_file="/content/drive/My Drive/pguard/thermal/train/labels/${file}.txt"
    if [ -f "$img_file" ] && [ -f "$label_file" ]; then
        mv "$img_file" "/content/drive/My Drive/pguard/thermal/val/images/"
        mv "$label_file" "/content/drive/My Drive/pguard/thermal/val/labels/"
        echo "Moved $file to validation"
    fi
done < /tmp/selected_files.txt

# Clean up
rm /tmp/selected_files.txt

# Verify counts
echo "Validation images: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/images/"*.png | wc -l)"  # Should be 350
echo "Validation labels: $(ls -1 "/content/drive/My Drive/pguard/thermal/val/labels/"*.txt | wc -l)"  # Should be 350
echo "Training images: $(ls -1 "/content/drive/My Drive/pguard/thermal/train/images/"*.png | wc -l)"  # Should be 1294 (1433 - 139)
echo "Training labels: $(find "/content/drive/My Drive/pguard/thermal/train/labels/" -type f -name "*.txt" | wc -l)"  # Should be 1703 (1842 - 139)

Moved TH_903 to validation
Moved TH_656 to validation
Moved TH_1805 to validation
Moved TH_1399 to validation
Moved TH_539 to validation
Moved TH_1450 to validation
Moved TH_1015 to validation
Moved TH_788 to validation
Moved TH_287 to validation
Moved TH_116 to validation
Moved TH_189 to validation
Moved TH_572 to validation
Moved TH_622 to validation
Moved TH_261 to validation
Moved TH_552 to validation
Moved TH_1436 to validation
Moved TH_631 to validation
Moved TH_747 to validation
Moved TH_779 to validation
Moved TH_913 to validation
Moved TH_830 to validation
Moved TH_50 to validation
Moved TH_1601 to validation
Moved TH_1515 to validation
Moved TH_701 to validation
Moved TH_1802 to validation
Moved TH_1093 to validation
Moved TH_172 to validation
Moved TH_149 to validation
Moved TH_1321 to validation
Moved TH_734 to validation
Moved TH_624 to validation
Moved TH_893 to validation
Moved TH_593 to validation
Moved TH_271 to validation
Moved TH_1434 to validation
Moved TH_79 to val

In [ ]:
import os
import shutil
import random

# Define directories
base_dir = "/content/drive/My Drive/pguard/thermal"
train_img_dir = os.path.join(base_dir, "train/images")
train_lbl_dir = os.path.join(base_dir, "train/labels")
val_img_dir = os.path.join(base_dir, "val/images")
val_lbl_dir = os.path.join(base_dir, "val/labels")
unpaired_dir = os.path.join(base_dir, "unpaired/images")

# Create unpaired directory if it doesn’t exist
os.makedirs(unpaired_dir, exist_ok=True)

# Helper function to get file names without extensions
def get_file_names(directory, extension):
    return {f.split('.')[0] for f in os.listdir(directory) if f.endswith(extension)}

# Function to generate a unique base name for paired files
def get_unique_base_name(dest_img_dir, dest_lbl_dir, base_name):
    counter = 0
    unique_name = base_name
    while os.path.exists(os.path.join(dest_img_dir, f"{unique_name}.png")) or \
          os.path.exists(os.path.join(dest_lbl_dir, f"{unique_name}.txt")):
        unique_name = f"{base_name}_{counter}"
        counter += 1
    return unique_name

# Function to generate a unique image name for unpaired images
def get_unique_image_name(dest_img_dir, base_name):
    counter = 0
    unique_name = base_name
    while os.path.exists(os.path.join(dest_img_dir, f"{unique_name}.png")):
        unique_name = f"{base_name}_{counter}"
        counter += 1
    return unique_name

# Step 1: Move unpaired images from validation to unpaired directory
val_images = get_file_names(val_img_dir, '.png')
val_labels = get_file_names(val_lbl_dir, '.txt')
unpaired_val_images = val_images - val_labels
for file in unpaired_val_images:
    shutil.move(os.path.join(val_img_dir, f"{file}.png"), unpaired_dir)

# Step 2: Move unpaired images from training to unpaired directory
train_images = get_file_names(train_img_dir, '.png')
train_labels = get_file_names(train_lbl_dir, '.txt')
unpaired_train_images = train_images - train_labels
for file in unpaired_train_images:
    shutil.move(os.path.join(train_img_dir, f"{file}.png"), unpaired_dir)

# Step 3: Consolidate paired files into training directory with unique names
paired_val_files = val_images & val_labels
for file in paired_val_files:
    unique_base_name = get_unique_base_name(train_img_dir, train_lbl_dir, file)
    shutil.move(os.path.join(val_img_dir, f"{file}.png"), os.path.join(train_img_dir, f"{unique_base_name}.png"))
    shutil.move(os.path.join(val_lbl_dir, f"{file}.txt"), os.path.join(train_lbl_dir, f"{unique_base_name}.txt"))

# Step 4: Refresh paired files list after consolidation
train_images = get_file_names(train_img_dir, '.png')
train_labels = get_file_names(train_lbl_dir, '.txt')
paired_files = list(train_images & train_labels)

# Step 5: Split paired files (70% training, 30% validation)
random.shuffle(paired_files)
val_size = 526  # Assuming total images = 1752, 30% ≈ 526
train_size = len(paired_files) - val_size
for file in paired_files[:val_size]:
    shutil.move(os.path.join(train_img_dir, f"{file}.png"), val_img_dir)
    shutil.move(os.path.join(train_lbl_dir, f"{file}.txt"), val_lbl_dir)

# Step 6: Move unpaired images back to training with unique names
for file in os.listdir(unpaired_dir):
    base_name = os.path.splitext(file)[0]
    unique_base_name = get_unique_image_name(train_img_dir, base_name)
    unique_file_name = f"{unique_base_name}.png"
    shutil.move(os.path.join(unpaired_dir, file), os.path.join(train_img_dir, unique_file_name))

# Verify counts
train_images_count = len(os.listdir(train_img_dir))
train_labels_count = len(os.listdir(train_lbl_dir))
val_images_count = len(os.listdir(val_img_dir))
val_labels_count = len(os.listdir(val_lbl_dir))

print(f"Training images: {train_images_count}")
print(f"Training labels: {train_labels_count}")
print(f"Validation images: {val_images_count}")
print(f"Validation labels: {val_labels_count}")

Training images: 1256
Training labels: 1248
Validation images: 526
Validation labels: 526


In [ ]:

!ls -1 "/content/drive/My Drive/pguard/thermal/"

thermal_labeled
thermal.yaml
train
val


In [ ]:
import os
import shutil

# Paths
thermal_train_images_dir = "/content/drive/My Drive/pguard/thermal/train/images"
thermal_train_labels_dir = "/content/drive/My Drive/pguard/thermal/train/labels"
thermal_labeled_train_images_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"
thermal_labeled_train_labels_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"

# Get lists of images
thermal_train_images = set(os.listdir(thermal_train_images_dir))
thermal_labeled_train_images = set(os.listdir(thermal_labeled_train_images_dir))

# Check if thermal_labeled images are a subset of thermal train images
missing_images = thermal_labeled_train_images - thermal_train_images

if missing_images:
    print(f"Found {len(missing_images)} images in thermal_labeled that are not in thermal train split. Copying them...")
    for img in missing_images:
        img_path = os.path.join(thermal_labeled_train_images_dir, img)
        label_filename = img.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
        label_path = os.path.join(thermal_labeled_train_labels_dir, label_filename)

        # Copy image
        shutil.copy(img_path, os.path.join(thermal_train_images_dir, img))

        # Copy label
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(thermal_train_labels_dir, label_filename))
        else:
            print(f"Warning: Label file {label_filename} not found for image {img}.")
    print("Copying completed.")
else:
    print("All images in thermal_labeled are already in thermal train split.")

All images in thermal_labeled are already in thermal train split.


In [ ]:
import os
import shutil
import random

# Paths
thermal_val_images_dir = "/content/drive/My Drive/pguard/thermal/val/images"
thermal_val_labels_dir = "/content/drive/My Drive/pguard/thermal/val/labels"
thermal_labeled_val_images_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/val/images"
thermal_labeled_val_labels_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/val/labels"

# Create directories if they don't exist
os.makedirs(thermal_labeled_val_images_dir, exist_ok=True)
os.makedirs(thermal_labeled_val_labels_dir, exist_ok=True)

# Check current val images
val_images = os.listdir(thermal_labeled_val_images_dir)

if len(val_images) < 10:
    print(f"Thermal labeled val split has {len(val_images)} images. Populating with 20 images from thermal val split...")

    # Get images from thermal val split
    thermal_val_images = [f for f in os.listdir(thermal_val_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(thermal_val_images)

    # Take 20 images
    selected_images = thermal_val_images[:20]

    # Move images and labels
    for img in selected_images:
        img_path = os.path.join(thermal_val_images_dir, img)
        label_filename = img.replace('.png', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt')
        label_path = os.path.join(thermal_val_labels_dir, label_filename)

        # Copy image
        shutil.copy(img_path, os.path.join(thermal_labeled_val_images_dir, img))

        # Copy label
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(thermal_labeled_val_labels_dir, label_filename))
        else:
            print(f"Warning: Label file {label_filename} not found for image {img}.")
    print("Val split populated.")
else:
    print(f"Thermal labeled val split already has {len(val_images)} images. No action needed.")

Thermal labeled val split already has 40 images. No action needed.


In [ ]:
!git clone https://github.com/davidjaw/Domain-adaptation-object-detection-with-YOLOv8.git
%cd Domain-adaptation-object-detection-with-YOLOv8

Cloning into 'Domain-adaptation-object-detection-with-YOLOv8'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 37 (delta 13), reused 36 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 32.71 KiB | 10.90 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/Domain-adaptation-object-detection-with-YOLOv8


In [ ]:
!pip install ultralytics pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
!pip install torch==2.0.1 torchvision==0.15.2 ultralytics pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.25.2
    Uninstalling numpy-1.25.2:
      Successfully uninstalled numpy-1.25.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
import os

# Create data directory
!mkdir -p data

# Write visible.yaml
with open('data/visible.yaml', 'w') as f:
    f.write("""path: /content/drive/My Drive/pguard
train: train/images
val: val/images
nc: 1
names: ['lp']
""")

# Write thermal.yaml
with open('data/thermal.yaml', 'w') as f:
    f.write("""path: /content/drive/My Drive/pguard/thermal
train: train/images
val: val/images
nc: 1
names: ['lp']
""")

# Write thermal_labeled.yaml
with open('data/thermal_labeled.yaml', 'w') as f:
    f.write("""path: /content/drive/My Drive/pguard/thermal/thermal_labeled
train: train/images
val: val/images
nc: 1
names: ['lp']
""")

In [ ]:
%mv /content/Domain-adaptation-object-detection-with-YOLOv8/Domain-adaptation-object-detection-with-YOLOv8/Domain-adaptation-object-detection-with-YOLOv8/Domain-adaptation-object-detection-with-YOLOv8/data/*.yaml /content/Domain-adaptation-object-detection-with-YOLOv8/data/


mv: cannot stat '/content/Domain-adaptation-object-detection-with-YOLOv8/Domain-adaptation-object-detection-with-YOLOv8/Domain-adaptation-object-detection-with-YOLOv8/Domain-adaptation-object-detection-with-YOLOv8/data/*.yaml': No such file or directory


In [ ]:
%rm -rf /content/Domain-adaptation-object-detection-with-YOLOv8/Domain-adaptation-object-detection-with-YOLOv8


In [ ]:
import os

model_path = "/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt"
if os.path.exists(model_path):
    print(f"Pre-trained model found at {model_path}")
else:
    print(f"Pre-trained model not found at {model_path}. Please check the path.")

Pre-trained model found at /content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py

from ultralytics.models import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, emojis, clean_url
import ultralytics.utils.callbacks.tensorboard as tb_module
import numpy as np
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from functools import partial
import random
import os
import shutil
import pandas as pd

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.add_callback('on_train_start', self._init_discriminator)

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)
    kwargs = dict(imgsz=640, epochs=40, val=True, workers=2, batch=16, seed=9527, patience=10, save=True, plots=True)

    # Clear previous outputs
    exp_name = 'visible_to_thermal_semisupervised'
    exp_dir = os.path.join(SCRIPT_DIR, 'runs', 'detect', exp_name)
    if os.path.isdir(exp_dir): shutil.rmtree(exp_dir)

    # Locate best model weights
    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    # Resolve data configs
    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    # Prepare trainer
    trainer = partial(CustomTrainer, target_domain_data_cfg=data_thermal, labeled_thermal_data_cfg=data_thermal_lbl)

    # Train & overwrite
    model.train(trainer=trainer, data=data_visible, name=exp_name, exist_ok=True, **kwargs)
    model.val(data=data_thermal, name='val_thermal', batch=8)

if __name__ == '__main__':
    main()



Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
from ultralytics.models import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, emojis, clean_url
import ultralytics.utils.callbacks.tensorboard as tb_module
import numpy as np
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from functools import partial
import random
import os
import shutil
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        return self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]

    @property
    def labels(self):
        # Combine labels from source and labeled datasets (target may not have labels)
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        LOGGER.info(f"Initializing CustomTrainer with data: {self.args.data}")
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.current_epoch = 0
        self.metrics = []  # List to store training metrics

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            # Create combined dataset
            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            # Adjust batch size to account for combined datasets
            per_dataset_batch = source_batch_size + target_batch_size
            if self.labeled_data:
                per_dataset_batch += labeled_batch_size

            # Create DataLoader
            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = DataLoader(
                dataset=combined_dataset,
                batch_size=per_dataset_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=self.trainset.collate_fn,
                drop_last=True,
            )
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def get_dataloader_target(self, batch_size, rank=-1):
        dataset = self.t_train
        sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(dataset, shuffle=True)
        return DataLoader(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=sampler is None,
            num_workers=self.args.workers,
            sampler=sampler,
            pin_memory=True,
            collate_fn=getattr(dataset, 'collate_fn', None),
            drop_last=True,
        )

    def get_dataloader_labeled(self, batch_size, rank=-1):
        dataset = self.labeled_train
        sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(dataset, shuffle=True)
        return DataLoader(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=sampler is None,
            num_workers=self.args.workers,
            sampler=sampler,
            pin_memory=True,
            collate_fn=getattr(dataset, 'collate_fn', None),
            drop_last=True,
        )

    def combine_loaders(self, *loaders):
        # This method is no longer used for train mode, kept for compatibility
        from itertools import cycle, chain
        return [list(chain(*batch)) for batch in zip(*loaders)]

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.1 * epoch)

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def forward_batch(self, batch, epoch=0):
        self.current_epoch = epoch
        self.update_consistency_weight(epoch)

        if self.labeled_data:
            bs = batch[0].shape[0] // 3
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:2*bs] for b in batch]
            labeled_batch = [b[2*bs:] for b in batch]
        else:
            bs = batch[0].shape[0] // 2
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:] for b in batch]
            labeled_batch = None

        # Supervised loss on source domain
        loss, loss_items = super().forward_batch(source_batch)
        supervised_loss = loss.item()

        # Get features from both domains
        with torch.no_grad():
            self.model.eval()
            _, source_feats = self.model(source_batch[0], return_feats=True)
            _, target_feats = self.model(target_batch[0], return_feats=True)
            self.model.train()

        # Consistency loss
        consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
        loss += self.consistency_weight * consistency_loss

        # Supervised loss on labeled target data if available
        labeled_loss = 0.0
        if labeled_batch:
            labeled_loss, _ = super().forward_batch(labeled_batch)
            loss += labeled_loss
            labeled_loss = labeled_loss.item()

        # Domain adversarial training
        domain_preds = self.discriminator(source_feats + target_feats)
        domain_labels = torch.cat([
            torch.zeros(len(source_feats[0]), device=self.device),
            torch.ones(len(target_feats[0]), device=self.device)
        ])
        adv_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        loss += adv_loss

        # Update discriminator
        self.discriminator.optim.zero_grad()
        adv_loss.backward(retain_graph=True)
        self.discriminator.optim.step()

        # Log metrics
        self.metrics.append({
            'epoch': epoch,
            'total_loss': loss.item(),
            'supervised_loss': supervised_loss,
            'consistency_loss': consistency_loss.item(),
            'adversarial_loss': adv_loss.item(),
            'labeled_loss': labeled_loss
        })

        loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
        return loss, loss_items

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)
    kwargs = dict(imgsz=640, epochs=40, val=True, workers=2, batch=16, seed=9527, patience=10, save=True, plots=True)

    # Define experiment name and local directory
    exp_name = 'visible_to_thermal_semisupervised'
    base_dir = '/content/pguard/domain_adaptation_process'
    exp_dir = os.path.join(base_dir, exp_name)

    # Clear experiment directory if exists
    if os.path.isdir(exp_dir):
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)

    # Locate best model weights
    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    # Resolve data configs
    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    # Prepare trainer
    trainer = partial(CustomTrainer, target_domain_data_cfg=data_thermal, labeled_thermal_data_cfg=data_thermal_lbl)

    # Train and save results
    results = model.train(trainer=trainer, data=data_visible, name=exp_dir, exist_ok=True, **kwargs)

    # Validate and save results
    val_results = model.val(data=data_thermal, name=os.path.join(base_dir, 'val_thermal'), batch=8)

    # Save metrics to CSV
    trainer_instance = results.trainer
    if trainer_instance.metrics:
        metrics_df = pd.DataFrame(trainer_instance.metrics)
        metrics_csv_path = os.path.join(exp_dir, 'training_metrics.csv')
        metrics_df.to_csv(metrics_csv_path, index=False)

if __name__ == '__main__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
import os
import shutil
import pandas as pd
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from functools import partial
from torch.utils.data import Dataset, DataLoader
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, emojis, clean_url
import ultralytics.utils.callbacks.tensorboard as tb_module
from ultralytics import YOLO

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        return self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]

    @property
    def labels(self):
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        LOGGER.info(f"Initializing CustomTrainer with data: {self.args.data}")
        # Initialize source dataset
        self.trainset, self.testset = self.get_dataset(data=self.args.data, require_labels=True)
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.adversarial_weight = 0.5
        self.current_epoch = 0
        self.metrics = []
        self.domain_metrics = []
        self.start_time = datetime.now()

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            per_dataset_batch = source_batch_size + target_batch_size
            if self.labeled_data:
                per_dataset_batch += labeled_batch_size

            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = DataLoader(
                dataset=combined_dataset,
                batch_size=per_dataset_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=self.trainset.collate_fn,
                drop_last=True,
            )
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def get_dataloader_target(self, batch_size, rank=-1):
        dataset = self.t_train
        sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(dataset, shuffle=True)
        return DataLoader(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=sampler is None,
            num_workers=self.args.workers,
            sampler=sampler,
            pin_memory=True,
            collate_fn=getattr(dataset, 'collate_fn', None),
            drop_last=True,
        )

    def get_dataloader_labeled(self, batch_size, rank=-1):
        dataset = self.labeled_train
        sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(dataset, shuffle=True)
        return DataLoader(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=sampler is None,
            num_workers=self.args.workers,
            sampler=sampler,
            pin_memory=True,
            collate_fn=getattr(dataset, 'collate_fn', None),
            drop_last=True,
        )

    def combine_loaders(self, *loaders):
        from itertools import cycle, chain
        return [list(chain(*batch)) for batch in zip(*loaders)]

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.05 * epoch)

    def forward_batch(self, batch, epoch=0):
        self.current_epoch = epoch
        self.update_consistency_weight(epoch)

        if self.labeled_data:
            bs = batch[0].shape[0] // 3
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:2*bs] for b in batch]
            labeled_batch = [b[2*bs:] for b in batch]
        else:
            bs = batch[0].shape[0] // 2
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:] for b in batch]
            labeled_batch = None

        loss, loss_items = super().forward_batch(source_batch)
        supervised_loss = loss.item()

        with torch.no_grad():
            self.model.eval()
            _, source_feats = self.model(source_batch[0], return_feats=True)
            _, target_feats = self.model(target_batch[0], return_feats=True)
            self.model.train()

        consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
        loss += self.consistency_weight * consistency_loss

        labeled_loss = 0.0
        if labeled_batch:
            labeled_loss, _ = super().forward_batch(labeled_batch)
            loss += 2.0 * labeled_loss
            labeled_loss = labeled_loss.item()

        domain_preds = self.discriminator(source_feats + target_feats)
        domain_labels = torch.cat([
            torch.zeros(len(source_feats[0]), device=self.device),
            torch.ones(len(target_feats[0]), device=self.device)
        ])
        adv_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        loss += self.adversarial_weight * adv_loss

        self.discriminator.optim.zero_grad()
        adv_loss.backward(retain_graph=True)
        self.discriminator.optim.step()

        domain_acc = ((domain_preds > 0).float() == domain_labels).float().mean().item()

        metrics = {
            'epoch': epoch,
            'total_loss': loss.item(),
            'supervised_loss': supervised_loss,
            'consistency_loss': consistency_loss.item(),
            'adversarial_loss': adv_loss.item(),
            'labeled_loss': labeled_loss,
            'domain_accuracy': domain_acc,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        self.metrics.append(metrics)
        self.domain_metrics.append({
            'epoch': epoch,
            'domain_accuracy': domain_acc,
            'adv_loss': adv_loss.item()
        })

        loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
        return loss, loss_items

    def on_train_epoch_end(self, *args, **kwargs):
        from ultralytics.utils import LOGGER
        if self.metrics:
            epoch_metrics = {
                'epoch': self.current_epoch,
                'total_loss': np.mean([m['total_loss'] for m in self.metrics[-self.args.batch:]]),
                'supervised_loss': np.mean([m['supervised_loss'] for m in self.metrics[-self.args.batch:]]),
                'consistency_loss': np.mean([m['consistency_loss'] for m in self.metrics[-self.args.batch:]]),
                'adversarial_loss': np.mean([m['adversarial_loss'] for m in self.metrics[-self.args.batch:]]),
                'labeled_loss': np.mean([m['labeled_loss'] for m in self.metrics[-self.args.batch:]]),
                'domain_accuracy': np.mean([m['domain_accuracy'] for m in self.metrics[-self.args.batch:]]),
            }
            self.metrics.append(epoch_metrics)
            self.metrics = [m for m in self.metrics if m['epoch'] == self.current_epoch]

            # Log losses for the current epoch
            if RANK in (-1, 0):
                LOGGER.info(f"\nEpoch {self.current_epoch}/{self.args.epochs}")
                LOGGER.info(f"Total Loss: {epoch_metrics['total_loss']:.4f}")
                LOGGER.info(f"Supervised Loss: {epoch_metrics['supervised_loss']:.4f}")
                LOGGER.info(f"Consistency Loss: {epoch_metrics['consistency_loss']:.4f}")
                LOGGER.info(f"Adversarial Loss: {epoch_metrics['adversarial_loss']:.4f}")
                LOGGER.info(f"Labeled Loss: {epoch_metrics['labeled_loss']:.4f}")
                LOGGER.info(f"Domain Accuracy: {epoch_metrics['domain_accuracy']:.4f}")

            # Perform validation on both source and target datasets every 20 epochs
            if self.current_epoch % 20 == 0 and self.current_epoch > 0:
                if RANK in (-1, 0):
                    LOGGER.info(f"\nValidating at epoch {self.current_epoch}...")
                    # Validate on source (visible) dataset
                    val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                    # Validate on target (thermal) dataset
                    thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

                    LOGGER.info(f"\nValidation Results at Epoch {self.current_epoch}:")
                    LOGGER.info(f"Source Domain (Visible): mAP50={val_results.box.map50:.4f}, mAP50-95={val_results.box.map:.4f}")
                    LOGGER.info(f"Target Domain (Thermal): mAP50={thermal_results.box.map50:.4f}, mAP50-95={thermal_results.box.map:.4f}")

                    self.thermal_metrics.append({
                        'epoch': self.current_epoch,
                        'source_mAP50': val_results.box.map50,
                        'source_mAP50-95': val_results.box.map,
                        'thermal_mAP50': thermal_results.box.map50,
                        'thermal_mAP50-95': thermal_results.box.map,
                    })

    def save_metrics(self):
        if RANK in (-1, 0):
            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)

            # Save training metrics
            if self.metrics:
                metrics_df = pd.DataFrame(self.metrics)
                metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                metrics_df.to_csv(metrics_path, index=False)
                LOGGER.info(f'Saved training metrics to {metrics_path}')

            # Save domain adaptation metrics
            if self.domain_metrics:
                domain_df = pd.DataFrame(self.domain_metrics)
                domain_path = os.path.join(metrics_dir, 'domain_metrics.csv')
                domain_df.to_csv(domain_path, index=False)
                LOGGER.info(f'Saved domain adaptation metrics to {domain_path}')

            # Save validation metrics
            if self.thermal_metrics:
                thermal_df = pd.DataFrame(self.thermal_metrics)
                thermal_path = os.path.join(metrics_dir, 'validation_metrics.csv')
                thermal_df.to_csv(thermal_path, index=False)
                LOGGER.info(f'Saved validation metrics to {thermal_path}')

    def on_train_end(self):
        self.save_metrics()

        if RANK in (-1, 0):
            # Validate on source (visible) dataset
            val_results = self.model.val(data=self.args.data, batch=self.args.batch)
            # Validate on target (thermal) dataset
            thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

            LOGGER.info(f"\nFinal Validation Results:")
            LOGGER.info(f"Source Domain (Visible): mAP50={val_results.box.map50:.4f}, mAP50-95={val_results.box.map:.4f}")
            LOGGER.info(f"Target Domain (Thermal): mAP50={thermal_results.box.map50:.4f}, mAP50-95={thermal_results.box.map:.4f}")

            duration = datetime.now() - self.start_time
            LOGGER.info(f"\nTraining completed in {duration}")

        return super().on_train_end()

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)

    kwargs = dict(
        imgsz=640,
        epochs=40,
        val=True,
        workers=4,
        batch=16,
        seed=9527,
        patience=30,
        save=True,
        plots=True,
        lr0=0.002,
        lrf=0.002,
        warmup_epochs=5,
        optimizer='AdamW',
        weight_decay=0.0005,
        momentum=0.9,
    )

    exp_name = 'visible_to_thermal_semisupervised'
    base_dir = '/content/pguard/domain_adaptation_process'
    exp_dir = os.path.join(base_dir, exp_name)

    if os.path.isdir(exp_dir):
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)

    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    trainer = partial(CustomTrainer, target_domain_data_cfg=data_thermal, labeled_thermal_data_cfg=data_thermal_lbl)

    results = model.train(
        trainer=trainer,
        data=data_visible,
        name=exp_dir,
        exist_ok=True,
        **kwargs
    )

    trainer_instance = results.trainer
    if trainer_instance.metrics:
        metrics_df = pd.DataFrame(trainer_instance.metrics)
        metrics_csv_path = os.path.join(exp_dir, 'training_metrics.csv')
        metrics_df.to_csv(metrics_csv_path, index=False)

    val_dir = os.path.join(base_dir, 'val_thermal')
    os.makedirs(val_dir, exist_ok=True)
    val_results = model.val(data=data_thermal, name=val_dir, batch=8)

    LOGGER.info("\nTraining completed successfully!")
    LOGGER.info(f"Results saved to: {exp_dir}")
    LOGGER.info(f"Validation results saved to: {val_dir}")

if __name__ == '__main__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
!pip install pyyaml

In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
import os
import shutil
import pandas as pd
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from torch.utils.data import Dataset, DataLoader
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import RANK, TQDM, colorstr, emojis, clean_url
from ultralytics import YOLO
import sys

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        return self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]

    @property
    def labels(self):
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        print(f"Initializing CustomTrainer with data: {self.args.data}")
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.training_metrics = []  # For batch-level metrics
        self.domain_metrics = []   # For domain adaptation metrics
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.adversarial_weight = 0.5
        self.current_epoch = 0
        self.start_time = datetime.now()
        self.batch_count = 0

        # Adjust training parameters
        self.args.patience = 40
        self.args.lr0 = 0.002
        self.args.lrf = 0.002
        self.args.warmup_epochs = 5

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            # Ensure self.trainset is a dataset object
            source_data = check_det_dataset(self.args.data)
            self.trainset, _ = self.get_dataset(data=source_data, require_labels=True)

            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            per_dataset_batch = source_batch_size + target_batch_size
            if self.labeled_data:
                per_dataset_batch += labeled_batch_size

            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = DataLoader(
                dataset=combined_dataset,
                batch_size=per_dataset_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=self.trainset.collate_fn,
                drop_last=True,
            )
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def forward_batch(self, batch, epoch=0):
        self.current_epoch = epoch
        self.update_consistency_weight(epoch)
        self.batch_count += 1

        if self.labeled_data:
            bs = batch[0].shape[0] // 3
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:2*bs] for b in batch]
            labeled_batch = [b[2*bs:] for b in batch]
        else:
            bs = batch[0].shape[0] // 2
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:] for b in batch]
            labeled_batch = None

        loss, loss_items = super().forward_batch(source_batch)
        supervised_loss = loss.item()

        with torch.no_grad():
            self.model.eval()
            _, source_feats = self.model(source_batch[0], return_feats=True)
            _, target_feats = self.model(target_batch[0], return_feats=True)
            self.model.train()

        consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
        loss += self.consistency_weight * consistency_loss

        labeled_loss = 0.0
        if labeled_batch:
            labeled_loss, _ = super().forward_batch(labeled_batch)
            loss += 2.0 * labeled_loss
            labeled_loss = labeled_loss.item()

        domain_preds = self.discriminator(source_feats + target_feats)
        domain_labels = torch.cat([
            torch.zeros(len(source_feats[0]), device=self.device),
            torch.ones(len(target_feats[0]), device=self.device)
        ])
        adv_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        loss += self.adversarial_weight * adv_loss

        self.discriminator.optim.zero_grad()
        adv_loss.backward(retain_graph=True)
        self.discriminator.optim.step()

        domain_acc = ((domain_preds > 0).float() == domain_labels).float().mean().item()

        metrics = {
            'epoch': epoch,
            'total_loss': loss.item(),
            'supervised_loss': supervised_loss,
            'consistency_loss': consistency_loss.item(),
            'adversarial_loss': adv_loss.item(),
            'labeled_loss': labeled_loss,
            'domain_accuracy': domain_acc,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        self.training_metrics.append(metrics)
        self.domain_metrics.append({
            'epoch': epoch,
            'domain_accuracy': domain_acc,
            'adv_loss': adv_loss.item()
        })

        if self.batch_count % 10 == 0:
            print(f"\n{'-'*40}")
            print(f"Epoch {self.current_epoch}, Batch {self.batch_count} - Batch Metrics")
            print(f"{'-'*40}")
            print(f"{'Metric':<20} {'Value':>10}")
            print(f"{'-'*30}")
            print(f"{'Total Loss':<20} {loss.item():>10.4f}")
            print(f"{'Supervised Loss':<20} {supervised_loss:>10.4f}")
            print(f"{'Consistency Loss':<20} {consistency_loss.item():>10.4f}")
            print(f"{'Adversarial Loss':<20} {adv_loss.item():>10.4f}")
            print(f"{'Labeled Loss': citerion.<20} {labeled_loss:>10.4f}")
            print(f"{'Domain Accuracy':<20} {domain_acc:>10.4f}")
            print(f"{'-'*40}")
            sys.stdout.flush()

        loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
        return loss, loss_items

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.05 * epoch)

    def save_metrics(self, metrics=None):
        # Call parent save_metrics to ensure results.csv is written
        super().save_metrics(metrics=metrics)

        if RANK in (-1, 0):
            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)

            # Append ultralytics metrics to self.training_metrics if provided
            if metrics:
                ultralytics_metrics = {
                    'epoch': self.current_epoch,
                    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                # Include relevant metrics from ultralytics (e.g., losses, mAP)
                for key, value in metrics.items():
                    if key.startswith('metrics/') or key in ['box_loss', 'cls_loss', 'dfl_loss', 'lr0', 'lr1', 'lr2']:
                        ultralytics_metrics[key] = value
                self.training_metrics.append(ultralytics_metrics)

            # Save training metrics
            if self.training_metrics:
                metrics_df = pd.DataFrame(self.training_metrics)
                metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                metrics_df.to_csv(metrics_path, index=False)
                print(f'Saved training metrics to {metrics_path}')

            # Save domain adaptation metrics
            if self.domain_metrics:
                domain_df = pd.DataFrame(self.domain_metrics)
                domain_path = os.path.join(metrics_dir, 'domain_metrics.csv')
                domain_df.to_csv(domain_path, index=False)
                print(f'Saved domain adaptation metrics to {domain_path}')

            # Save validation metrics
            if self.thermal_metrics:
                thermal_df = pd.DataFrame(self.thermal_metrics)
                thermal_path = os.path.join(metrics_dir, 'validation_metrics.csv')
                thermal_df.to_csv(thermal_path, index=False)
                print(f'Saved validation metrics to {thermal_path}')

    def on_train_epoch_end(self):
        if self.current_epoch % 20 == 0 and self.current_epoch > 0:
            print(f"\n{'='*50}")
            print(f"Validation Triggered at Epoch {self.current_epoch}")
            print(f"{'='*50}")
            sys.stdout.flush()
            try:
                print("Validating Source Domain (Visible)...")
                sys.stdout.flush()
                val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                print("Validating Target Domain (Thermal)...")
                sys.stdout.flush()
                thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

                print(f"\n{'-'*50}")
                print(f"Validation Results at Epoch {self.current_epoch}")
                print(f"{'-'*50}")
                print(f"Source Domain (Visible):")
                print(f"  mAP50: {val_results.box.map50:.4f}")
                print(f"  mAP50-95: {val_results.box.map:.4f}")
                print(f"Target Domain (Thermal):")
                print(f"  mAP50: {thermal_results.box.map50:.4f}")
                print(f"  mAP50-95: {thermal_results.box.map:.4f}")
                print(f"{'='*50}")
                sys.stdout.flush()

                self.thermal_metrics.append({
                    'epoch': self.current_epoch,
                    'source_mAP50': val_results.box.map50,
                    'source_mAP50-95': val_results.box.map,
                    'thermal_mAP50': thermal_results.box.map50,
                    'thermal_mAP50-95': thermal_results.box.map,
                })
            except Exception as e:
                print(f"Validation failed at epoch {self.current_epoch}: {e}")
                sys.stdout.flush()

    def on_train_end(self):
        self.save_metrics()

        if RANK in (-1, 0):
            val_results = self.model.val(data=self.args.data, batch=self.args.batch)
            thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

            print(f"\nFinal Validation Results:")
            print(f"Source Domain (Visible): mAP50={val_results.box.map50:.4f}, mAP50-95={val_results.box.map:.4f}")
            print(f"Target Domain (Thermal): mAP50={thermal_results.box.map50:.4f}, mAP50-95={thermal_results.box.map:.4f}")

            duration = datetime.now() - self.start_time
            print(f"\nTraining completed in {duration}")

        return super().on_train_end()

class CustomTrainerFactory:
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg):
        self.target_domain_data_cfg = target_domain_data_cfg
        self.labeled_thermal_data_cfg = labeled_thermal_data_cfg

    def __call__(self, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        return CustomTrainer(
            target_domain_data_cfg=self.target_domain_data_cfg,
            labeled_thermal_data_cfg=self.labeled_thermal_data_cfg,
            overrides=overrides,
            _callbacks=_callbacks
        )

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)

    kwargs = dict(
        imgsz=640,
        epochs=40,
        val=True,
        workers=4,
        batch=16,
        seed=9527,
        patience=40,
        save=True,
        plots=True,
        lr0=0.002,
        lrf=0.002,
        warmup_epochs=5,
        optimizer='AdamW',
        weight_decay=0.0005,
        momentum=0.9,
        data='data/visible.yaml'
    )

    exp_name = 'visible_to_thermal_semisupervised'
    base_dir = '/content/pguard/domain_adaptation_process'
    exp_dir = os.path.join(base_dir, exp_name)

    if os.path.isdir(exp_dir):
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)

    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    trainer = CustomTrainerFactory(
        target_domain_data_cfg=data_thermal,
        labeled_thermal_data_cfg=data_thermal_lbl
    )

    results = model.train(
        trainer=trainer,
        name=exp_dir,
        exist_ok=True,
        **kwargs
    )

    trainer_instance = results.trainer
    if trainer_instance.training_metrics:
        metrics_df = pd.DataFrame(trainer_instance.training_metrics)
        metrics_csv_path = os.path.join(exp_dir, 'training_metrics.csv')
        metrics_df.to_csv(metrics_csv_path, index=False)
        print(f'Saved training metrics to {metrics_csv_path}')

    val_dir = os.path.join(base_dir, 'val_thermal')
    os.makedirs(val_dir, exist_ok=True)
    val_results = model.val(data=data_thermal, name=val_dir, batch=8)

    print("\nTraining completed successfully!")
    print(f"Results saved to: {exp_dir}")
    print(f"Validation results saved to: {val_dir}")

if __name__ == '__main__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
import os
import shutil
import pandas as pd
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from torch.utils.data import Dataset, DataLoader
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import RANK, TQDM, colorstr, emojis, clean_url
from ultralytics import YOLO
import sys

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        return self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]

    @property
    def labels(self):
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        print(f"Initializing CustomTrainer with data: {self.args.data}")
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.training_metrics = []  # For batch-level metrics
        self.domain_metrics = []   # For domain adaptation metrics
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.adversarial_weight = 0.5
        self.current_epoch = 0
        self.start_time = datetime.now()
        self.batch_count = 0

        # Adjust training parameters
        self.args.patience = 40
        self.args.lr0 = 0.002
        self.args.lrf = 0.002
        self.args.warmup_epochs = 5

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            # Ensure self.trainset is a dataset object
            source_data = check_det_dataset(self.args.data)
            self.trainset, _ = self.get_dataset(data=source_data, require_labels=True)

            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            per_dataset_batch = source_batch_size + target_batch_size
            if self.labeled_data:
                per_dataset_batch += labeled_batch_size

            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = DataLoader(
                dataset=combined_dataset,
                batch_size=per_dataset_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=self.trainset.collate_fn,
                drop_last=True,
            )
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def forward_batch(self, batch, epoch=0):
        self.current_epoch = epoch
        self.update_consistency_weight(epoch)
        self.batch_count += 1

        print(f"Debug: Batch {self.batch_count}, Epoch {self.current_epoch}, Total Batch Size: {batch[0].shape[0]}")
        sys.stdout.flush()

        if self.labeled_data:
            bs = batch[0].shape[0] // 3
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:2*bs] for b in batch]
            labeled_batch = [b[2*bs:] for b in batch]
            print(f"Debug: Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}, Labeled Batch Size: {labeled_batch[0].shape[0]}")
        else:
            bs = batch[0].shape[0] // 2
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:] for b in batch]
            labeled_batch = None
            print(f"Debug: Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}")
        sys.stdout.flush()

        loss, loss_items = super().forward_batch(source_batch)
        supervised_loss = loss.item()

        # Compute thermal loss for target batch (unlabeled)
        thermal_loss = 0.0
        with torch.no_grad():
            self.model.eval()
            _, source_feats = self.model(source_batch[0], return_feats=True)
            _, target_feats = self.model(target_batch[0], return_feats=True)
            self.model.train()

        consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
        loss += self.consistency_weight * consistency_loss

        labeled_loss = 0.0
        if labeled_batch:
            labeled_loss, _ = super().forward_batch(labeled_batch)
            loss += 2.0 * labeled_loss
            labeled_loss = labeled_loss.item()

        domain_preds = self.discriminator(source_feats + target_feats)
        domain_labels = torch.cat([
            torch.zeros(len(source_feats[0]), device=self.device),
            torch.ones(len(target_feats[0]), device=self.device)
        ])
        adv_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        loss += self.adversarial_weight * adv_loss

        self.discriminator.optim.zero_grad()
        adv_loss.backward(retain_graph=True)
        self.discriminator.optim.step()

        domain_acc = ((domain_preds > 0).float() == domain_labels).float().mean().item()

        metrics = {
            'epoch': epoch,
            'total_loss': loss.item(),
            'supervised_loss': supervised_loss,
            'thermal_loss': thermal_loss,
            'consistency_loss': consistency_loss.item(),
            'adversarial_loss': adv_loss.item(),
            'labeled_loss': labeled_loss,
            'domain_accuracy': domain_acc,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        self.training_metrics.append(metrics)
        self.domain_metrics.append({
            'epoch': epoch,
            'domain_accuracy': domain_acc,
            'adv_loss': adv_loss.item()
        })

        # Print metrics every batch for debugging
        print(f"\n{'-'*40}")
        print(f"Epoch {self.current_epoch}, Batch {self.batch_count} - Batch Metrics")
        print(f"{'-'*40}")
        print(f"{'Metric':<20} {'Value':>10}")
        print(f"{'-'*30}")
        print(f"{'Total Loss':<20} {loss.item():>10.4f}")
        print(f"{'Supervised Loss':<20} {supervised_loss:>10.4f}")
        print(f"{'Thermal Loss':<20} {thermal_loss:>10.4f}")
        print(f"{'Consistency Loss':<20} {consistency_loss.item():>10.4f}")
        print(f"{'Adversarial Loss':<20} {adv_loss.item():>10.4f}")
        print(f"{'Labeled Loss':<20} {labeled_loss:>10.4f}")
        print(f"{'Domain Accuracy':<20} {domain_acc:>10.4f}")
        print(f"{'-'*40}")
        sys.stdout.flush()

        loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
        return loss, loss_items

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.05 * epoch)

    def save_metrics(self, metrics=None):
        # Call parent save_metrics to ensure results.csv is written
        super().save_metrics(metrics=metrics)

        if RANK in (-1, 0):
            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)

            # Append ultralytics metrics to self.training_metrics if provided
            if metrics:
                ultralytics_metrics = {
                    'epoch': self.current_epoch,
                    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                # Include relevant metrics from ultralytics (e.g., losses, mAP)
                for key, value in metrics.items():
                    if key.startswith('metrics/') or key in ['box_loss', 'cls_loss', 'dfl_loss', 'lr0', 'lr1', 'lr2']:
                        ultralytics_metrics[key] = value
                self.training_metrics.append(ultralytics_metrics)

            # Include thermal validation metrics in training_metrics
            if self.thermal_metrics:
                for thermal_metric in self.thermal_metrics:
                    if thermal_metric['epoch'] == self.current_epoch:
                        self.training_metrics.append({
                            'epoch': self.current_epoch,
                            'source_mAP50': thermal_metric['source_mAP50'],
                            'source_mAP50-95': thermal_metric['source_mAP50-95'],
                            'thermal_mAP50': thermal_metric['thermal_mAP50'],
                            'thermal_mAP50-95': thermal_metric['thermal_mAP50-95'],
                            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                        })

            # Save training metrics
            if self.training_metrics:
                metrics_df = pd.DataFrame(self.training_metrics)
                metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                metrics_df.to_csv(metrics_path, index=False)
                print(f'Saved training metrics to {metrics_path}')

            # Save domain adaptation metrics
            if self.domain_metrics:
                domain_df = pd.DataFrame(self.domain_metrics)
                domain_path = os.path.join(metrics_dir, 'domain_metrics.csv')
                domain_df.to_csv(domain_path, index=False)
                print(f'Saved domain adaptation metrics to {domain_path}')

            # Save validation metrics
            if self.thermal_metrics:
                thermal_df = pd.DataFrame(self.thermal_metrics)
                thermal_path = os.path.join(metrics_dir, 'validation_metrics.csv')
                thermal_df.to_csv(thermal_path, index=False)
                print(f'Saved validation metrics to {thermal_path}')

    def on_train_epoch_end(self):
        if self.current_epoch % 20 == 0 and self.current_epoch > 0:
            print(f"\n{'='*50}")
            print(f"Validation Triggered at Epoch {self.current_epoch}")
            print(f"{'='*50}")
            sys.stdout.flush()
            try:
                print("Validating Source Domain (Visible)...")
                sys.stdout.flush()
                val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                print("Validating Target Domain (Thermal)...")
                sys.stdout.flush()
                thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

                print(f"\n{'-'*50}")
                print(f"Validation Results at Epoch {self.current_epoch}")
                print(f"{'-'*50}")
                print(f"Source Domain (Visible):")
                print(f"  mAP50: {val_results.box.map50:.4f}")
                print(f"  mAP50-95: {val_results.box.map:.4f}")
                print(f"Target Domain (Thermal):")
                print(f"  mAP50: {thermal_results.box.map50:.4f}")
                print(f"  mAP50-95: {thermal_results.box.map:.4f}")
                print(f"{'='*50}")
                sys.stdout.flush()

                self.thermal_metrics.append({
                    'epoch': self.current_epoch,
                    'source_mAP50': val_results.box.map50,
                    'source_mAP50-95': val_results.box.map,
                    'thermal_mAP50': thermal_results.box.map50,
                    'thermal_mAP50-95': thermal_results.box.map,
                })
            except Exception as e:
                print(f"Validation failed at epoch {self.current_epoch}: {e}")
                sys.stdout.flush()
        # Reset batch count at the end of each epoch
        self.batch_count = 0

    def on_train_end(self):
        self.save_metrics()

        if RANK in (-1, 0):
            val_results = self.model.val(data=self.args.data, batch=self.args.batch)
            thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

            print(f"\nFinal Validation Results:")
            print(f"Source Domain (Visible): mAP50={val_results.box.map50:.4f}, mAP50-95={val_results.box.map:.4f}")
            print(f"Target Domain (Thermal): mAP50={thermal_results.box.map50:.4f}, mAP50-95={thermal_results.box.map:.4f}")

            duration = datetime.now() - self.start_time
            print(f"\nTraining completed in {duration}")

        return super().on_train_end()

class CustomTrainerFactory:
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg):
        self.target_domain_data_cfg = target_domain_data_cfg
        self.labeled_thermal_data_cfg = labeled_thermal_data_cfg

    def __call__(self, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        return CustomTrainer(
            target_domain_data_cfg=self.target_domain_data_cfg,
            labeled_thermal_data_cfg=self.labeled_thermal_data_cfg,
            overrides=overrides,
            _callbacks=_callbacks
        )

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)

    kwargs = dict(
        imgsz=640,
        epochs=40,
        val=True,
        workers=4,
        batch=16,
        seed=9527,
        patience=40,
        save=True,
        plots=True,
        lr0=0.002,
        lrf=0.002,
        warmup_epochs=5,
        optimizer='AdamW',
        weight_decay=0.0005,
        momentum=0.9,
        data='data/visible.yaml'
    )

    exp_name = 'visible_to_thermal_semisupervised'
    base_dir = '/content/pguard/domain_adaptation_process'
    exp_dir = os.path.join(base_dir, exp_name)

    if os.path.isdir(exp_dir):
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)

    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    trainer = CustomTrainerFactory(
        target_domain_data_cfg=data_thermal,
        labeled_thermal_data_cfg=data_thermal_lbl
    )

    results = model.train(
        trainer=trainer,
        name=exp_dir,
        exist_ok=True,
        **kwargs
    )

    trainer_instance = results.trainer
    if trainer_instance.training_metrics:
        metrics_df = pd.DataFrame(trainer_instance.training_metrics)
        metrics_csv_path = os.path.join(exp_dir, 'training_metrics.csv')
        metrics_df.to_csv(metrics_path, index=False)
        print(f'Saved training metrics to {metrics_csv_path}')

    val_dir = os.path.join(base_dir, 'val_thermal')
    os.makedirs(val_dir, exist_ok=True)
    val_results = model.val(data=data_thermal, name=val_dir, batch=8)

    print("\nTraining completed successfully!")
    print(f"Results saved to: {exp_dir}")
    print(f"Validation results saved to: {val_dir}")

if __name__ == '__main__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
from ultralytics.utils import LOGGER
logging.getLogger('ultralytics').setLevel(logging.INFO)
LOGGER.setLevel(logging.INFO)

In [ ]:
!cp -r /content/Domain-adaptation-object-detection-with-YOLOv8 \
      /content/drive/MyDrive/Domain-adaptation-object-detection-with-YOLOv8


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
import os
import shutil
import pandas as pd
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from torch.utils.data import Dataset, DataLoader
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import RANK, TQDM, colorstr, emojis, clean_url, LOGGER, __version__ as ultralytics_version
from ultralytics import YOLO
import sys
import logging
from ultralytics.nn.tasks import v8DetectionLoss

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

# Set LOGGER to DEBUG level for detailed logging
logging.getLogger('ultralytics').setLevel(logging.DEBUG)
LOGGER.setLevel(logging.DEBUG)

# Add file handler for logging
def setup_file_logging(log_dir):
    os.makedirs(log_dir, exist_ok=True)
    file_handler = logging.FileHandler(os.path.join(log_dir, 'training_log.txt'))
    file_handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    LOGGER.addHandler(file_handler)

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        item = self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]
        # Handle different dataset output formats
        if isinstance(item, (list, tuple)):
            return item
        else:
            return item['img'], item.get('bboxes', torch.tensor([])), item['im_file'], item['resized_shape']

    @property
    def labels(self):
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class ResettableDataLoader(DataLoader):
    def reset(self):
        self._iterator = iter(self)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        LOGGER.debug(f"Ultralytics version: {ultralytics_version}")
        LOGGER.debug(f"Initializing CustomTrainer with data: {self.args.data}, amp: {self.args.amp}")
        sys.stdout.flush()
        self.amp = self.args.amp
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.training_metrics = []
        self.domain_metrics = []
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.adversarial_weight = 0.5
        self.current_epoch = 0
        self.start_time = datetime.now()
        self.batch_count = 0

        # Adjust training parameters
        self.args.patience = 40
        self.args.lr0 = 0.002
        self.args.lrf = 0.002
        self.args.warmup_epochs = 5

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            source_data = check_det_dataset(self.args.data)
            self.trainset, _ = self.get_dataset(data=source_data, require_labels=True)

            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            def custom_collate_fn(batch):
                imgs, targets, paths, shapes = [], [], [], []
                for item in batch:
                    img, target, path, shape = item
                    # Convert image to float and normalize to [0, 1]
                    if img.dtype == torch.uint8:
                        img = img.float() / 255.0
                    imgs.append(img)
                    # Ensure targets are properly formatted
                    if target is None or len(target) == 0:
                        target = torch.tensor([], dtype=torch.float32).reshape(0, 6)  # Empty targets for unlabeled data
                    targets.append(target)
                    paths.append(path)
                    shapes.append(shape)
                return (
                    torch.stack(imgs),
                    targets,  # Keep as list to handle variable sizes
                    paths,
                    torch.tensor(shapes)
                )

            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = ResettableDataLoader(
                dataset=combined_dataset,
                batch_size=total_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=custom_collate_fn,
                drop_last=True,
            )

            LOGGER.debug(f"DataLoader created with batch_size={total_batch}")
            try:
                sample_batch = next(iter(loader))
                LOGGER.debug(f"Sample batch structure: {type(sample_batch)}, contents: {[type(x) for x in sample_batch]}")
            except Exception as e:
                LOGGER.error(f"Failed to fetch sample batch from DataLoader: {str(e)}")
            sys.stdout.flush()
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def compute_yolo_loss(self, batch):
        """Compute YOLO detection loss for a batch."""
        imgs, targets, _, _ = batch
        imgs = imgs.to(self.device, non_blocking=True).float()  # Convert to float32

        # Handle empty targets
        if isinstance(targets, list):
            targets = [t.to(self.device, non_blocking=True) if len(t) > 0 else torch.tensor([], dtype=torch.float32, device=self.device).reshape(0, 6) for t in targets]
        else:
            targets = targets.to(self.device, non_blocking=True) if len(targets) > 0 else torch.tensor([], dtype=torch.float32, device=self.device).reshape(0, 6)

        # Forward pass
        with torch.cuda.amp.autocast(enabled=self.amp):
            preds = self.model(imgs)

        # Initialize loss function
        loss_fn = v8DetectionLoss(self.model)

        # Compute loss
        if isinstance(targets, list):
            loss, loss_items = 0, [0, 0, 0]
            valid_batches = 0
            for i, t in enumerate(targets):
                if len(t) > 0:  # Only compute loss for non-empty targets
                    l, li = loss_fn(preds, (imgs[i:i+1], t))
                    loss += l
                    loss_items = [x + y for x, y in zip(loss_items, li)]
                    valid_batches += 1
            if valid_batches > 0:
                loss /= valid_batches
                loss_items = [x / valid_batches for x in loss_items]
        else:
            loss, loss_items = loss_fn(preds, (imgs, targets))

        return loss, loss_items

    def train_step(self, batch):
        """Process a single training batch."""
        LOGGER.debug(f"Entering train_step for epoch {self.current_epoch}, batch {self.batch_count + 1}")
        sys.stdout.flush()
        loss, loss_items = self.forward_batch(batch, self.current_epoch)
        LOGGER.debug(f"Exiting train_step with loss: {float(loss.item()):.4f}")
        sys.stdout.flush()
        return loss, loss_items

    def _do_train(self, world_size=1):
        """Override _do_train to process batches with train_step."""
        LOGGER.debug(f"Entering _do_train with world_size={world_size}")
        sys.stdout.flush()

        try:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            self.model.to(self.device)
            self.model.train()

            # Initialize optimizer and scaler
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.args.lr0,
                weight_decay=self.args.weight_decay,
                betas=(self.args.momentum, 0.999)
            )
            LOGGER.debug(f"Optimizer initialized: AdamW(lr={self.args.lr0}, weight_decay={self.args.weight_decay}, betas=({self.args.momentum}, 0.999))")
            sys.stdout.flush()
            self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
            LOGGER.debug(f"Gradient scaler initialized with amp={self.amp}")
            sys.stdout.flush()

            # Get DataLoader
            train_loader = self.get_dataloader(self.trainset, self.args.batch, mode='train')
            nb = len(train_loader)
            LOGGER.debug(f"Number of batches per epoch: {nb}")
            sys.stdout.flush()

            for epoch in range(self.args.epochs):
                self.current_epoch = epoch
                self.batch_count = 0
                self.update_consistency_weight(epoch)  # Update consistency weight
                LOGGER.info(f"\nEpoch {epoch+1}/{self.args.epochs}")
                sys.stdout.flush()

                pbar = TQDM(enumerate(train_loader), total=nb)
                self.model.train()
                for i, batch in pbar:
                    LOGGER.debug(f"Processing batch {i+1}/{nb}, batch type: {type(batch)}")
                    sys.stdout.flush()

                    # Forward and backward pass
                    with torch.cuda.amp.autocast(enabled=self.amp):
                        loss, loss_items = self.train_step(batch)

                    # Backward pass
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()

                    # Update progress bar
                    pbar.set_description(f"Epoch {epoch+1}/{self.args.epochs} GPU_mem {torch.cuda.memory_allocated(self.device)/1E9:.2f}G "
                                        f"box_loss {loss_items[0]:.4f} cls_loss {loss_items[1]:.4f} dfl_loss {loss_items[2]:.4f}")

                # End of epoch validation
                if self.args.val:
                    metrics = self.validate()
                    self.save_metrics(metrics)
                    LOGGER.info(f"Validation metrics: {metrics}")
                    sys.stdout.flush()

                # Reset DataLoader
                train_loader.reset()

        except Exception as e:
            LOGGER.error(f"Error in _do_train: {str(e)}")
            sys.stdout.flush()
            raise

        LOGGER.debug(f"Exiting _do_train after {self.args.epochs} epochs")
        sys.stdout.flush()

    def forward_batch(self, batch, epoch=0):
        LOGGER.debug(f"Entering forward_batch for epoch {epoch}, batch {self.batch_count + 1}")
        sys.stdout.flush()

        try:
            # Ensure batch is in correct format
            imgs, targets, paths, shapes = batch
            LOGGER.debug(f"Batch {self.batch_count + 1}, Epoch {self.current_epoch}, Total Batch Size: {imgs.shape[0]}")
            sys.stdout.flush()

            # Dynamic batch splitting
            total_bs = imgs.shape[0]
            if self.labeled_data:
                bs = total_bs // 3
                source_batch = (imgs[:bs], targets[:bs], paths[:bs], shapes[:bs])
                target_batch = (imgs[bs:2*bs], targets[bs:2*bs], paths[bs:2*bs], shapes[bs:2*bs])
                labeled_batch = (imgs[2*bs:], targets[2*bs:], paths[2*bs:], shapes[2*bs:])
                LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}, Labeled Batch Size: {labeled_batch[0].shape[0]}")
            else:
                bs = total_bs // 2
                source_batch = (imgs[:bs], targets[:bs], paths[:bs], shapes[:bs])
                target_batch = (imgs[bs:], targets[bs:], paths[bs:], shapes[bs:])
                labeled_batch = None
                LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}")
            sys.stdout.flush()

            # Compute supervised loss on source domain
            loss, loss_items = self.compute_yolo_loss(source_batch)
            supervised_loss = loss.item() if loss != 0 else 0.0

            # Compute features for domain adaptation
            with torch.no_grad():
                self.model.eval()
                _, source_feats = self.model(source_batch[0], return_feats=True)
                _, target_feats = self.model(target_batch[0], return_feats=True)
                self.model.train()

            # Consistency loss
            consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
            loss += self.consistency_weight * consistency_loss

            # Supervised loss on labeled target data
            labeled_loss = 0.0
            if labeled_batch:
                labeled_loss, _ = self.compute_yolo_loss(labeled_batch)
                loss += 2.0 * labeled_loss
                labeled_loss = labeled_loss.item() if labeled_loss != 0 else 0.0

            # Domain adversarial loss
            # Combine features correctly for discriminator
            combined_feats = [torch.cat([sf, tf], dim=0) for sf, tf in zip(source_feats, target_feats)]
            domain_preds = self.discriminator(combined_feats)
            domain_labels = torch.cat([
                torch.zeros(bs, device=self.device),
                torch.ones(bs, device=self.device)
            ]).to(self.device)
            adv_loss = F.binary_cross_entropy_with_logits(domain_preds.squeeze(), domain_labels)
            loss += self.adversarial_weight * adv_loss

            # Update discriminator
            self.discriminator.optim.zero_grad()
            adv_loss.backward(retain_graph=True)
            self.discriminator.optim.step()

            # Compute domain accuracy
            domain_acc = ((domain_preds.squeeze() > 0).float() == domain_labels).float().mean().item()

            # Log metrics
            metrics = {
                'epoch': epoch,
                'total_loss': loss.item(),
                'supervised_loss': supervised_loss,
                'consistency_loss': consistency_loss.item(),
                'adversarial_loss': adv_loss.item(),
                'labeled_loss': labeled_loss,
                'domain_accuracy': domain_acc,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'box_loss': loss_items[0],
                'cls_loss': loss_items[1],
                'dfl_loss': loss_items[2],
            }
            self.training_metrics.append(metrics)
            self.domain_metrics.append({
                'epoch': epoch,
                'domain_accuracy': domain_acc,
                'adv_loss': adv_loss.item()
            })

            # Save metrics after each batch
            if RANK in (-1, 0):
                metrics_dir = os.path.join(self.save_dir, 'metrics')
                os.makedirs(metrics_dir, exist_ok=True)
                metrics_df = pd.DataFrame(self.training_metrics)
                metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                metrics_df.to_csv(metrics_path, index=False)
                LOGGER.debug(f'Saved batch metrics to {metrics_path}')
                sys.stdout.flush()

            # Print metrics every 10 batches to reduce logging overhead
            if self.batch_count % 10 == 0:
                log_message = (
                    f"Epoch {self.current_epoch}, Batch {self.batch_count + 1} - Batch Metrics\n"
                    f"{'-'*40}\n"
                    f"Total Loss: {loss.item():.4f}\n"
                    f"Supervised Loss: {supervised_loss:.4f}\n"
                    f"Consistency Loss: {consistency_loss.item():.4f}\n"
                    f"Adversarial Loss: {adv_loss.item():.4f}\n"
                    f"Labeled Loss: {labeled_loss:.4f}\n"
                    f"Domain Accuracy: {domain_acc:.4f}\n"
                    f"Box Loss: {loss_items[0]:.4f}\n"
                    f"Cls Loss: {loss_items[1]:.4f}\n"
                    f"Dfl Loss: {loss_items[2]:.4f}"
                )
                LOGGER.info(log_message)
                sys.stdout.flush()

            LOGGER.debug(f"Exiting forward_batch with loss: {loss.item():.4f}")
            sys.stdout.flush()
            self.batch_count += 1

            loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
            return loss, loss_items

        except RuntimeError as re:
            LOGGER.error(f"RuntimeError in forward_batch: {str(re)}")
            sys.stdout.flush()
            raise
        except Exception as e:
            LOGGER.error(f"Unexpected error in forward_batch: {str(e)}")
            sys.stdout.flush()
            raise

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.05 * epoch)

    def save_metrics(self, metrics=None):
        super().save_metrics(metrics=metrics)

        if RANK in (-1, 0):
            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)

            # Save training metrics
            if self.training_metrics:
                metrics_df = pd.DataFrame(self.training_metrics)
                metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                metrics_df.to_csv(metrics_path, index=False)
                LOGGER.info(f'Saved training metrics to {metrics_path}')
                sys.stdout.flush()

            # Save domain metrics
            if self.domain_metrics:
                domain_df = pd.DataFrame(self.domain_metrics)
                domain_path = os.path.join(metrics_dir, 'domain_metrics.csv')
                domain_df.to_csv(domain_path, index=False)
                LOGGER.info(f'Saved domain adaptation metrics to {domain_path}')
                sys.stdout.flush()

            # Save validation metrics
            if self.thermal_metrics:
                thermal_df = pd.DataFrame(self.thermal_metrics)
                thermal_path = os.path.join(metrics_dir, 'validation_metrics.csv')
                thermal_df.to_csv(thermal_path, index=False)
                LOGGER.info(f'Saved validation metrics to {thermal_path}')
                sys.stdout.flush()

    def on_train_epoch_end(self):
        # Validate every 5 epochs to better monitor domain adaptation
        if self.current_epoch % 5 == 0 and self.current_epoch > 0:
            LOGGER.info(f"\n{'='*50}")
            LOGGER.info(f"Validation Triggered at Epoch {self.current_epoch}")
            LOGGER.info(f"{'='*50}")
            sys.stdout.flush()
            try:
                LOGGER.info("Validating Source Domain (Visible)...")
                sys.stdout.flush()
                val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                LOGGER.info("Validating Target Domain (Thermal)...")
                sys.stdout.flush()
                thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

                LOGGER.info(f"\n{'-'*50}")
                LOGGER.info(f"Validation Results at Epoch {self.current_epoch}")
                LOGGER.info(f"{'-'*50}")
                LOGGER.info(f"Source Domain (Visible):")
                LOGGER.info(f"  mAP50: {val_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {val_results.box.map:.4f}")
                LOGGER.info(f"Target Domain (Thermal):")
                LOGGER.info(f"  mAP50: {thermal_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {thermal_results.box.map:.4f}")
                LOGGER.info(f"{'='*50}")
                sys.stdout.flush()

                self.thermal_metrics.append({
                    'epoch': self.current_epoch,
                    'source_mAP50': val_results.box.map50,
                    'source_mAP50-95': val_results.box.map,
                    'thermal_mAP50': thermal_results.box.map50,
                    'thermal_mAP50-95': thermal_results.box.map,
                })
            except Exception as e:
                LOGGER.error(f"Validation failed at epoch {self.current_epoch}: {e}")
                sys.stdout.flush()
        self.batch_count = 0

    def on_train_end(self):
        self.save_metrics()

        if RANK in (-1, 0):
            # Final validation
            LOGGER.info("\nRunning final validation...")
            sys.stdout.flush()

            # Validate on source domain
            LOGGER.info("Validating Source Domain (Visible)...")
            sys.stdout.flush()
            val_results = self.model.val(data=self.args.data, batch=self.args.batch)

            # Validate on target domain
            LOGGER.info("Validating Target Domain (Thermal)...")
            sys.stdout.flush()
            thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

            LOGGER.info(f"\nFinal Validation Results:")
            LOGGER.info(f"Source Domain (Visible): mAP50={val_results.box.map50:.4f}, mAP50-95={val_results.box.map:.4f}")
            LOGGER.info(f"Target Domain (Thermal): mAP50={thermal_results.box.map50:.4f}, mAP50-95={thermal_results.box.map:.4f}")
            sys.stdout.flush()

            # Save final validation results
            final_metrics = {
                'source_mAP50': val_results.box.map50,
                'source_mAP50-95': val_results.box.map,
                'thermal_mAP50': thermal_results.box.map50,
                'thermal_mAP50-95': thermal_results.box.map,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)
            final_metrics_path = os.path.join(metrics_dir, 'final_validation.csv')
            pd.DataFrame([final_metrics]).to_csv(final_metrics_path, index=False)
            LOGGER.info(f"Saved final validation results to {final_metrics_path}")
            sys.stdout.flush()

            # Print average losses per epoch
            metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
            if os.path.isfile(metrics_path):
                metrics_df = pd.read_csv(metrics_path)
                avg_metrics = metrics_df.groupby('epoch').mean(numeric_only=True)
                LOGGER.info(f"\n{'='*50}")
                LOGGER.info("Average Losses Per Epoch")
                LOGGER.info(f"{'-'*50}")
                for epoch in avg_metrics.index:
                    LOGGER.info(f"Epoch {int(epoch)}:")
                    LOGGER.info(f"  Total Loss: {avg_metrics.loc[epoch, 'total_loss']:.4f}")
                    LOGGER.info(f"  Supervised Loss: {avg_metrics.loc[epoch, 'supervised_loss']:.4f}")
                    LOGGER.info(f"  Consistency Loss: {avg_metrics.loc[epoch, 'consistency_loss']:.4f}")
                    LOGGER.info(f"  Adversarial Loss: {avg_metrics.loc[epoch, 'adversarial_loss']:.4f}")
                    LOGGER.info(f"  Labeled Loss: {avg_metrics.loc[epoch, 'labeled_loss']:.4f}")
                    LOGGER.info(f"  Domain Accuracy: {avg_metrics.loc[epoch, 'domain_accuracy']:.4f}")
                LOGGER.info(f"{'='*50}")
                sys.stdout.flush()

            duration = datetime.now() - self.start_time
            LOGGER.info(f"\nTraining completed in {duration}")
            sys.stdout.flush()

        return super().on_train_end()

class CustomTrainerFactory:
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg):
        self.target_domain_data_cfg = target_domain_data_cfg
        self.labeled_thermal_data_cfg = labeled_thermal_data_cfg

    def __call__(self, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        return CustomTrainer(
            target_domain_data_cfg=self.target_domain_data_cfg,
            labeled_thermal_data_cfg=self.labeled_thermal_data_cfg,
            overrides=overrides,
            _callbacks=_callbacks
        )

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)

    kwargs = dict(
        imgsz=640,
        epochs=40,
        val=True,
        workers=4,
        batch=16,
        seed=9527,
        patience=40,
        save=True,
        plots=True,
        lr0=0.002,
        lrf=0.002,
        warmup_epochs=5,
        optimizer='AdamW',
        weight_decay=0.0005,
        momentum=0.9,
        data='data/visible.yaml'
    )

    exp_name = 'visible_to_thermal_semisupervised'
    base_dir = '/content/pguard/domain_adaptation_process'
    exp_dir = os.path.join(base_dir, exp_name)

    os.makedirs(base_dir, exist_ok=True)
    if os.path.isdir(exp_dir):
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)

    # Setup file logging
    setup_file_logging(exp_dir)

    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    trainer = CustomTrainerFactory(
        target_domain_data_cfg=data_thermal,
        labeled_thermal_data_cfg=data_thermal_lbl
    )

    results = model.train(
        trainer=trainer,
        name=exp_dir,
        exist_ok=True,
        **kwargs
    )

    # Final thermal dataset evaluation
    LOGGER.info("\nRunning final evaluation on thermal dataset...")
    sys.stdout.flush()

    val_dir = os.path.join(base_dir, 'val_thermal')
    os.makedirs(val_dir, exist_ok=True)
    val_results = model.val(data=data_thermal, name=val_dir, batch=8)

    LOGGER.info("\nThermal Dataset Evaluation Results:")
    LOGGER.info(f"  mAP50: {val_results.box.map50:.4f}")
    LOGGER.info(f"  mAP50-95: {val_results.box.map:.4f}")
    sys.stdout.flush()

    LOGGER.info("\nTraining completed successfully!")
    LOGGER.info(f"Results saved to: {exp_dir}")
    LOGGER.info(f"Validation results saved to: {val_dir}")
    sys.stdout.flush()

if __name__ == '__module__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
import shutil
import pandas as pd
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from torch.utils.data import Dataset, DataLoader
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import RANK, TQDM, colorstr, emojis, clean_url, LOGGER, __version__ as ultralytics_version
from ultralytics import YOLO
import sys
import logging
from ultralytics.nn.tasks import v8DetectionLoss
import os

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

# Set LOGGER to INFO for console, DEBUG for file
logging.getLogger('ultralytics').setLevel(logging.INFO)
LOGGER.setLevel(logging.DEBUG)  # DEBUG for file to capture all details

# Add file and console handlers for logging
def setup_file_logging(log_dir):
    os.makedirs(log_dir, exist_ok=True)
    # File handler for detailed debugging
    file_handler = logging.FileHandler(os.path.join(log_dir, 'training_log.txt'))
    file_handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    LOGGER.addHandler(file_handler)

    # Console handler for user-visible output
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)
    LOGGER.addHandler(console_handler)

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64, 128, 256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        item = self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]
        if isinstance(item, (list, tuple)) and len(item) >= 4:
            img, target, path, shape = item[:4]
            if isinstance(target, torch.Tensor) and target.shape[-1] >= 5:
                LOGGER.debug(f"CombinedDataset: Loaded {path} with target shape {target.shape}")
            else:
                LOGGER.warning(f"Invalid target format for {path}: {target.shape if isinstance(target, torch.Tensor) else type(target)}, resetting to empty")
                target = torch.zeros((0, 6), dtype=torch.float32)
        else:
            LOGGER.warning(f"Unexpected item format for idx {idx}: {type(item)}, resetting to empty")
            img = item.get('img', torch.zeros((3, 640, 640), dtype=torch.float32))
            target = torch.zeros((0, 6), dtype=torch.float32)
            path = item.get('im_file', 'unknown')
            shape = item.get('resized_shape', (640, 640))
        return img, target, path, shape

    @property
    def labels(self):
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class ResettableDataLoader(DataLoader):
    def reset(self):
        self._iterator = iter(self)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, overrides=None, _callbacks=None):
        self.target_domain_data_cfg = target_domain_data_cfg
        self.labeled_thermal_data_cfg = labeled_thermal_data_cfg
        super().__init__(overrides=overrides, _callbacks=_callbacks)
        LOGGER.info(f"Ultralytics version: {ultralytics_version}")
        LOGGER.info(f"Initializing CustomTrainer with data: {self.args.data}, amp: {self.args.amp}")
        sys.stdout.flush()
        self.amp = self.args.amp
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.training_metrics = []
        self.domain_metrics = []
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.adversarial_weight = 0.5
        self.current_epoch = 0
        self.start_time = datetime.now()
        self.batch_count = 0

        # Adjust training parameters
        self.args.patience = 40
        self.args.lr0 = 0.002
        self.args.lrf = 0.002
        self.args.warmup_epochs = 5

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            source_data = check_det_dataset(self.args.data)
            self.trainset, _ = self.get_dataset(data=source_data, require_labels=True)

            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            def custom_collate_fn(batch):
                imgs, targets, paths, shapes = [], [], [], []
                for batch_idx, item in enumerate(batch):
                    img, target, path, shape = item
                    if img.dtype == torch.uint8:
                        img = img.float() / 255.0
                    imgs.append(img)
                    if target is None or len(target) == 0:
                        target = torch.zeros((0, 6), dtype=torch.float32)
                    elif isinstance(target, torch.Tensor):
                        if target.shape[-1] == 5:  # YOLO format [class_id, x_center, y_center, width, height]
                            batch_indices = torch.full((target.shape[0], 1), batch_idx, dtype=torch.float32)
                            target = torch.cat([batch_indices, target], dim=1)
                        elif target.shape[-1] != 6:
                            LOGGER.warning(f"Invalid target shape {target.shape} for {path}, resetting to empty")
                            target = torch.zeros((0, 6), dtype=torch.float32)
                    else:
                        LOGGER.warning(f"Non-tensor target for {path}: {type(target)}, resetting to empty")
                        target = torch.zeros((0, 6), dtype=torch.float32)
                    targets.append(target)
                    paths.append(path)
                    shapes.append(shape)
                stacked_imgs = torch.stack(imgs)
                LOGGER.debug(f"Collate_fn output: img_shape={stacked_imgs.shape}, img_dtype={stacked_imgs.dtype}, target_shapes={[t.shape for t in targets]}, paths={paths}")
                return (
                    stacked_imgs,
                    targets,
                    paths,
                    torch.tensor(shapes, dtype=torch.float32)
                )

            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = ResettableDataLoader(
                dataset=combined_dataset,
                batch_size=total_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=custom_collate_fn,
                drop_last=True,
            )

            LOGGER.info(f"DataLoader created with batch_size={total_batch}")
            try:
                sample_batch = next(iter(loader))
                LOGGER.info(f"Sample batch: img_shape={sample_batch[0].shape}, target_shapes={[t.shape for t in sample_batch[1]]}")
            except Exception as e:
                LOGGER.error(f"Failed to fetch sample batch from DataLoader: {str(e)}")
                raise
            sys.stdout.flush()
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def compute_yolo_loss(self, batch):
        """Compute YOLO detection loss for a batch."""
        imgs, targets, paths, _ = batch
        imgs = imgs.to(self.device, non_blocking=True).float()
        LOGGER.debug(f"compute_yolo_loss: img_shape={imgs.shape}, img_dtype={imgs.dtype}, target_shapes={[t.shape for t in targets]}, paths={paths}")

        targets = [
            t.to(self.device, non_blocking=True) if len(t) > 0 else torch.zeros((0, 6), dtype=torch.float32, device=self.device)
            for t in targets
        ]

        with torch.cuda.amp.autocast(enabled=self.amp):
            preds = self.model(imgs)
            LOGGER.debug(f"Predictions: {type(preds)}, {len(preds)} items")

        loss_fn = v8DetectionLoss(self.model)

        try:
            loss, loss_items = loss_fn(preds, targets)
            LOGGER.debug(f"Loss: {loss.item()}, Loss Items: {loss_items}")
        except Exception as e:
            LOGGER.error(f"Error in v8DetectionLoss: {str(e)}")
            raise

        return loss, loss_items

    def train_step(self, batch):
        """Process a single training batch."""
        LOGGER.debug(f"Entering train_step for epoch {self.current_epoch}, batch {self.batch_count + 1}")
        loss, loss_items = self.forward_batch(batch, self.current_epoch)
        LOGGER.debug(f"Exiting train_step with loss: {float(loss.item()):.4f}")
        return loss, loss_items

    def _do_train(self, world_size=1):
        """Override _do_train to process batches with train_step."""
        LOGGER.info(f"Entering _do_train with world_size={world_size}")
        sys.stdout.flush()

        try:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            self.model.to(self.device)
            self.model.train()

            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.args.lr0,
                weight_decay=self.args.weight_decay,
                betas=(self.args.momentum, 0.999)
            )
            LOGGER.info(f"Optimizer initialized: AdamW(lr={self.args.lr0}, weight_decay={self.args.weight_decay})")
            self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
            LOGGER.info(f"Gradient scaler initialized with amp={self.amp}")

            train_loader = self.get_dataloader(self.trainset, self.args.batch, mode='train')
            nb = len(train_loader)
            LOGGER.info(f"Number of batches per epoch: {nb}")

            for epoch in range(self.args.epochs):
                self.current_epoch = epoch
                self.batch_count = 0
                self.update_consistency_weight(epoch)
                LOGGER.info(f"\nEpoch {epoch+1}/{self.args.epochs}")

                pbar = TQDM(enumerate(train_loader), total=nb)
                self.model.train()
                for i, batch in pbar:
                    LOGGER.debug(f"Processing batch {i+1}/{nb}, batch type: {type(batch)}")

                    with torch.cuda.amp.autocast(enabled=self.amp):
                        loss, loss_items = self.train_step(batch)

                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()

                    pbar.set_description(f"Epoch {epoch+1}/{self.args.epochs} GPU_mem {torch.cuda.memory_allocated(self.device)/1E9:.2f}G "
                                        f"total_loss {loss.item():.4f} box_loss {loss_items[0]:.4f} cls_loss {loss_items[1]:.4f} dfl_loss {loss_items[2]:.4f}")

                if self.args.val:
                    try:
                        metrics = self.validate()
                        self.save_metrics(metrics)
                        LOGGER.info(f"Validation metrics: {metrics}")
                    except Exception as e:
                        LOGGER.warning(f"Validation failed: {str(e)}")

                train_loader.reset()

        except Exception as e:
            LOGGER.error(f"Error in _do_train: {str(e)}")
            raise

        LOGGER.info(f"Exiting _do_train after {self.args.epochs} epochs")

    def forward_batch(self, batch, epoch=0):
        LOGGER.debug(f"Entering forward_batch for epoch {epoch}, batch {self.batch_count + 1}")
        try:
            imgs, targets, paths, shapes = batch
            total_bs = imgs.shape[0]
            LOGGER.debug(f"Batch {self.batch_count + 1}, Epoch {epoch}, Total Batch Size: {total_bs}, img_dtype: {imgs.dtype}")

            if self.labeled_data:
                bs = total_bs // 3
                source_end = bs
                target_end = 2 * bs
                source_batch = (imgs[:source_end], targets[:source_end], paths[:source_end], shapes[:source_end])
                target_batch = (imgs[source_end:target_end], targets[source_end:target_end], paths[source_end:target_end], shapes[source_end:target_end])
                labeled_batch = (imgs[target_end:], targets[target_end:], paths[target_end:], shapes[target_end:])
                LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}, Labeled Batch Size: {labeled_batch[0].shape[0]}")
            else:
                bs = total_bs // 2
                source_batch = (imgs[:bs], targets[:bs], paths[:bs], shapes[:bs])
                target_batch = (imgs[bs:], targets[bs:], paths[bs:], shapes[:bs])
                labeled_batch = None
                LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}")

            loss, loss_items = self.compute_yolo_loss(source_batch)
            supervised_loss = loss.item() if loss != 0 else 0.0

            with torch.no_grad():
                self.model.eval()
                try:
                    _, source_feats = self.model(source_batch[0], return_feats=True)
                    _, target_feats = self.model(target_batch[0], return_feats=True)
                except Exception as e:
                    LOGGER.error(f"Error extracting features: {str(e)}")
                    raise
                self.model.train()

            for sf, tf in zip(source_feats, target_feats):
                if sf.shape != tf.shape:
                    LOGGER.error(f"Feature dimension mismatch: source={sf.shape}, target={tf.shape}")
                    raise ValueError("Feature dimension mismatch")

            consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
            loss += self.consistency_weight * consistency_loss

            labeled_loss = 0.0
            if labeled_batch:
                labeled_loss, _ = self.compute_yolo_loss(labeled_batch)
                loss += 2.0 * labeled_loss
                labeled_loss = labeled_loss.item() if labeled_loss != 0 else 0.0

            combined_feats = [torch.cat([sf, tf], dim=0) for sf, tf in zip(source_feats, target_feats)]
            try:
                domain_preds = self.discriminator(combined_feats)
            except Exception as e:
                LOGGER.error(f"Error in discriminator forward pass: {str(e)}")
                raise
            domain_labels = torch.cat([
                torch.zeros(bs, device=self.device),
                torch.ones(bs, device=self.device)
            ])
            adv_loss = F.binary_cross_entropy_with_logits(domain_preds.squeeze(), domain_labels)
            loss += self.adversarial_weight * adv_loss

            self.discriminator.optim.zero_grad()
            adv_loss.backward(retain_graph=True)
            self.discriminator.optim.step()

            domain_acc = ((domain_preds.squeeze() > 0).float() == domain_labels).float().mean().item()

            metrics = {
                'epoch': epoch,
                'total_loss': loss.item(),
                'supervised_loss': supervised_loss,
                'consistency_loss': consistency_loss.item(),
                'adversarial_loss': adv_loss.item(),
                'labeled_loss': labeled_loss,
                'domain_accuracy': domain_acc,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'box_loss': float(loss_items[0]),
                'cls_loss': float(loss_items[1]),
                'dfl_loss': float(loss_items[2]),
            }
            self.training_metrics.append(metrics)
            self.domain_metrics.append({
                'epoch': epoch,
                'domain_accuracy': domain_acc,
                'adv_loss': adv_loss.item()
            })

            if RANK in (-1, 0):
                metrics_dir = os.path.join(self.save_dir, 'metrics')
                os.makedirs(metrics_dir, exist_ok=True)
                try:
                    metrics_df = pd.DataFrame(self.training_metrics)
                    metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                    metrics_df.to_csv(metrics_path, index=False)
                    LOGGER.debug(f'Saved batch metrics to {metrics_path}')
                except Exception as e:
                    LOGGER.warning(f"Failed to save metrics: {str(e)}")

            # Display all metrics on console for every batch
            log_message = (
                f"Epoch {epoch+1}, Batch {self.batch_count + 1} - Batch Metrics\n"
                f"{'-'*40}\n"
                f"Total Loss: {loss.item():.4f}\n"
                f"Supervised Loss: {supervised_loss:.4f}\n"
                f"Consistency Loss: {consistency_loss.item():.4f}\n"
                f"Adversarial Loss: {adv_loss.item():.4f}\n"
                f"Labeled Loss: {labeled_loss:.4f}\n"
                f"Domain Accuracy: {domain_acc:.4f}\n"
                f"Box Loss: {loss_items[0]:.4f}\n"
                f"Cls Loss: {loss_items[1]:.4f}\n"
                f"Dfl Loss: {loss_items[2]:.4f}"
            )
            LOGGER.info(log_message)

            LOGGER.debug(f"Exiting forward_batch with loss: {loss.item():.4f}")
            self.batch_count += 1
            loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
            return loss, loss_items

        except Exception as e:
            LOGGER.error(f"Error in forward_batch: {str(e)}")
            raise

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.05 * epoch)

    def save_metrics(self, metrics=None):
        super().save_metrics(metrics=metrics)
        if RANK in (-1, 0):
            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)
            try:
                if self.training_metrics:
                    metrics_df = pd.DataFrame(self.training_metrics)
                    metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                    metrics_df.to_csv(metrics_path, index=False)
                    LOGGER.info(f'Saved training metrics to {metrics_path}')
                if self.domain_metrics:
                    domain_df = pd.DataFrame(self.domain_metrics)
                    domain_path = os.path.join(metrics_dir, 'domain_metrics.csv')
                    domain_df.to_csv(domain_path, index=False)
                    LOGGER.info(f'Saved domain metrics to {domain_path}')
                if self.thermal_metrics:
                    thermal_df = pd.DataFrame(self.thermal_metrics)
                    thermal_path = os.path.join(metrics_dir, 'validation_metrics.csv')
                    thermal_df.to_csv(thermal_path, index=False)
                    LOGGER.info(f'Saved validation metrics to {thermal_path}')
            except Exception as e:
                LOGGER.warning(f"Failed to save metrics: {str(e)}")

    def on_train_epoch_end(self):
        if self.current_epoch % 5 == 0 and self.current_epoch > 0:
            LOGGER.info(f"\n{'='*50}\nValidation Triggered at Epoch {self.current_epoch}\n{'='*50}")
            try:
                LOGGER.info("Validating Source Domain (Visible)...")
                val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                LOGGER.info("Validating Target Domain (Thermal)...")
                thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)
                LOGGER.info(f"\n{'-'*50}\nValidation Results at Epoch {self.current_epoch}\n{'-'*50}")
                LOGGER.info(f"Source Domain (Visible):")
                LOGGER.info(f"  mAP50: {val_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {val_results.box.map:.4f}")
                LOGGER.info(f"  Precision: {val_results.box.p:.4f}")
                LOGGER.info(f"  Recall: {val_results.box.r:.4f}")
                LOGGER.info(f"Target Domain (Thermal):")
                LOGGER.info(f"  mAP50: {thermal_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {thermal_results.box.map:.4f}")
                LOGGER.info(f"  Precision: {thermal_results.box.p:.4f}")
                LOGGER.info(f"  Recall: {thermal_results.box.r:.4f}")
                LOGGER.info(f"{'='*50}")
                self.thermal_metrics.append({
                    'epoch': self.current_epoch,
                    'source_mAP50': val_results.box.map50,
                    'source_mAP50-95': val_results.box.map,
                    'source_precision': val_results.box.p,
                    'source_recall': val_results.box.r,
                    'thermal_mAP50': thermal_results.box.map50,
                    'thermal_mAP50-95': thermal_results.box.map,
                    'thermal_precision': thermal_results.box.p,
                    'thermal_recall': thermal_results.box.r,
                })
            except Exception as e:
                LOGGER.warning(f"Validation failed at epoch {self.current_epoch}: {str(e)}")
        self.batch_count = 0

    def on_train_end(self):
        self.save_metrics()
        if RANK in (-1, 0):
            LOGGER.info("\nRunning final validation...")
            try:
                LOGGER.info("Validating Source Domain (Visible)...")
                val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                LOGGER.info("Validating Target Domain (Thermal)...")
                thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)
                LOGGER.info(f"\n{'='*50}\nFinal Validation Results\n{'-'*50}")
                LOGGER.info(f"Source Domain (Visible):")
                LOGGER.info(f"  mAP50: {val_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {val_results.box.map:.4f}")
                LOGGER.info(f"  Precision: {val_results.box.p:.4f}")
                LOGGER.info(f"  Recall: {val_results.box.r:.4f}")
                LOGGER.info(f"Target Domain (Thermal):")
                LOGGER.info(f"  mAP50: {thermal_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {thermal_results.box.map:.4f}")
                LOGGER.info(f"  Precision: {thermal_results.box.p:.4f}")
                LOGGER.info(f"  Recall: {thermal_results.box.r:.4f}")
                LOGGER.info(f"{'='*50}")

                final_metrics = {
                    'source_mAP50': val_results.box.map50,
                    'source_mAP50-95': val_results.box.map,
                    'source_precision': val_results.box.p,
                    'source_recall': val_results.box.r,
                    'thermal_mAP50': thermal_results.box.map50,
                    'thermal_mAP50-95': thermal_results.box.map,
                    'thermal_precision': thermal_results.box.p,
                    'thermal_recall': thermal_results.box.r,
                    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                metrics_dir = os.path.join(self.save_dir, 'metrics')
                os.makedirs(metrics_dir, exist_ok=True)
                final_metrics_path = os.path.join(metrics_dir, 'final_validation.csv')
                pd.DataFrame([final_metrics]).to_csv(final_metrics_path, index=False)
                LOGGER.info(f"Saved final validation results to {final_metrics_path}")
            except Exception as e:
                LOGGER.warning(f"Final validation failed: {str(e)}")

            metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
            if os.path.isfile(metrics_path):
                try:
                    metrics_df = pd.read_csv(metrics_path)
                    avg_metrics = metrics_df.groupby('epoch').mean(numeric_only=True)
                    LOGGER.info(f"\n{'='*50}\nAverage Losses Per Epoch\n{'-'*50}")
                    for epoch in avg_metrics.index:
                        LOGGER.info(f"Epoch {int(epoch)}:")
                        LOGGER.info(f"  Total Loss: {avg_metrics.loc[epoch, 'total_loss']:.4f}")
                        LOGGER.info(f"  Supervised Loss: {avg_metrics.loc[epoch, 'supervised_loss']:.4f}")
                        LOGGER.info(f"  Consistency Loss: {avg_metrics.loc[epoch, 'consistency_loss']:.4f}")
                        LOGGER.info(f"  Adversarial Loss: {avg_metrics.loc[epoch, 'adversarial_loss']:.4f}")
                        LOGGER.info(f"  Labeled Loss: {avg_metrics.loc[epoch, 'labeled_loss']:.4f}")
                        LOGGER.info(f"  Domain Accuracy: {avg_metrics.loc[epoch, 'domain_accuracy']:.4f}")
                    LOGGER.info(f"{'='*50}")
                except Exception as e:
                    LOGGER.warning(f"Failed to compute average metrics: {str(e)}")

            duration = datetime.now() - self.start_time
            LOGGER.info(f"\nTraining completed in {duration}")

        super().on_train_end()

class CustomTrainerFactory:
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg, model, kwargs):
        self.target_domain_data_cfg = target_domain_data_cfg
        self.labeled_thermal_data_cfg = labeled_thermal_data_cfg
        self.model = model
        self.kwargs = kwargs

    def __call__(self, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        combined_overrides = self.kwargs.copy()
        combined_overrides.update(overrides)
        combined_overrides['model'] = self.model
        return CustomTrainer(
            target_domain_data_cfg=self.target_domain_data_cfg,
            labeled_thermal_data_cfg=self.labeled_thermal_data_cfg,
            overrides=combined_overrides,
            _callbacks=_callbacks
        )

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p):
            LOGGER.info(f"Found YAML file at: {p}")
            return p
    raise FileNotFoundError(f"YAML file not found at {rel_path}")

def main():
    try:
        LOGGER.info("Starting main() function")
        seed_everything(9527)

        cfg = {
            'model': '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt',
            'data': 'data/visible.yaml',
            'imgsz': 640,
            'epochs': 40,
            'val': True,
            'workers': 4,
            'batch': 12,
            'seed': 9527,
            'patience': 40,
            'save': True,
            'plots': True,
            'lr0': 0.002,
            'lrf': 0.002,
            'warmup_epochs': 5,
            'optimizer': 'AdamW',
            'weight_decay': 0.0005,
            'momentum': 0.9,
            'box': 7.5,
            'cls': 0.5,
            'dfl': 1.5,
            'kobj': 1.0,
            'amp': True,
            'cache': False,
            'single_cls': False
        }

        exp_name = 'visible_to_thermal_semisupervised'
        base_dir = '/content/pguard/domain_adaptation_process'
        exp_dir = os.path.join(base_dir, exp_name)

        os.makedirs(base_dir, exist_ok=True)
        if os.path.isdir(exp_dir):
            shutil.rmtree(exp_dir)
        os.makedirs(exp_dir, exist_ok=True)

        setup_file_logging(exp_dir)
        LOGGER.info(f"Experiment directory: {exp_dir}")

        model_path = cfg['model']
        if not os.path.isfile(model_path):
            w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
            model_path = w_local if os.path.isfile(w_local) else model_path
            if not os.path.isfile(model_path):
                raise FileNotFoundError(f"best.pt not found at {model_path} or {w_local}")
        LOGGER.info(f"Loading model from: {model_path}")
        try:
            model = YOLO(model_path)
        except Exception as e:
            LOGGER.error(f"Failed to load model: {str(e)}")
            raise

        try:
            data_visible = resolve_yaml(cfg['data'])
            data_thermal = resolve_yaml('data/thermal.yaml')
            data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')
            LOGGER.info(f"Dataset YAML files: visible={data_visible}, thermal={data_thermal}, thermal_labeled={data_thermal_lbl}")
        except FileNotFoundError as e:
            LOGGER.error(f"Dataset loading failed: {str(e)}")
            raise

        trainer = CustomTrainerFactory(
            target_domain_data_cfg=data_thermal,
            labeled_thermal_data_cfg=data_thermal_lbl,
            model=model,
            kwargs=cfg
        )

        LOGGER.info("Starting model training...")
        results = model.train(
            trainer=trainer,
            name=exp_dir,
            exist_ok=True,
            **cfg
        )

        LOGGER.info("\nRunning final evaluation on thermal dataset...")
        val_dir = os.path.join(base_dir, 'val_thermal')
        os.makedirs(val_dir, exist_ok=True)
        try:
            val_results = model.val(data=data_thermal, name=val_dir, batch=cfg['batch'])
            LOGGER.info("\nThermal Dataset Final Evaluation Results:")
            LOGGER.info(f"  mAP50: {val_results.box.map50:.4f}")
            LOGGER.info(f"  mAP50-95: {val_results.box.map:.4f}")
            LOGGER.info(f"  Precision: {val_results.box.p:.4f}")
            LOGGER.info(f"  Recall: {val_results.box.r:.4f}")
        except Exception as e:
            LOGGER.warning(f"Final evaluation failed: {str(e)}")

        LOGGER.info("\nTraining completed successfully!")
        LOGGER.info(f"Results saved to: {exp_dir}")
        LOGGER.info(f"Validation results saved to: {val_dir}")

    except Exception as e:
        LOGGER.error(f"Error in main(): {str(e)}")
        raise

if __name__ == '__main__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
import os
import glob

def check_annotations(label_dir):
    invalid_files = []
    for label_file in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(label_file, 'r') as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                invalid_files.append((label_file, len(parts)))
                break
            try:
                class_id, x_center, y_center, width, height = map(float, parts)
                if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 <= width <= 1 and 0 <= height <= 1):
                    invalid_files.append((label_file, "Values out of range"))
                    break
            except ValueError:
                invalid_files.append((label_file, "Invalid float values"))
                break
    return invalid_files

label_dirs = [
    "/content/drive/My Drive/pguard/train/labels",
    "/content/drive/My Drive/pguard/val/labels",
    "/content/drive/My Drive/pguard/thermal/train/labels",
    "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels",
]

for label_dir in label_dirs:
    print(f"Checking {label_dir}...")
    invalid_files = check_annotations(label_dir)
    if invalid_files:
        print(f"Found {len(invalid_files)} invalid files:")
        for file, issue in invalid_files:
            print(f"  {file}: {issue}")
    else:
        print("All files are valid.")

Checking /content/drive/My Drive/pguard/train/labels...
All files are valid.
Checking /content/drive/My Drive/pguard/val/labels...
All files are valid.
Checking /content/drive/My Drive/pguard/thermal/train/labels...
All files are valid.
Checking /content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels...
All files are valid.


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
import os
import shutil
import pandas as pd
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from torch.utils.data import Dataset, DataLoader
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import RANK, TQDM, colorstr, emojis, clean_url, LOGGER
from ultralytics import YOLO
import sys
import logging
from ultralytics.nn.tasks import v8DetectionLoss

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

# Ensure LOGGER is set to INFO level for visibility
logging.getLogger('ultralytics').setLevel(logging.INFO)
LOGGER.setLevel(logging.INFO)

# Add file handler for logging
def setup_file_logging(log_dir):
    os.makedirs(log_dir, exist_ok=True)
    file_handler = logging.FileHandler(os.path.join(log_dir, 'training_log.txt'))
    file_handler.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    LOGGER.addHandler(file_handler)

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        return self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]

    @property
    def labels(self):
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class ResettableDataLoader(DataLoader):
    def reset(self):
        # Reset the iterator (simulating Ultralytics' InfiniteDataLoader behavior)
        self._iterator = iter(self)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        LOGGER.info(f"Initializing CustomTrainer with data: {self.args.data}")
        sys.stdout.flush()
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
        self.additional_models, self.thermal_metrics = [], []
        self.training_metrics = []  # For batch-level metrics
        self.domain_metrics = []   # For domain adaptation metrics
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.adversarial_weight = 0.5
        self.current_epoch = 0
        self.start_time = datetime.now()
        self.batch_count = 0

        # Adjust training parameters
        self.args.patience = 50
        self.args.lr0 = 0.002
        self.args.lrf = 0.002
        self.args.warmup_epochs = 5

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            # Ensure self.trainset is a dataset object
            source_data = check_det_dataset(self.args.data)
            self.trainset, _ = self.get_dataset(data=source_data, require_labels=True)

            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            per_dataset_batch = source_batch_size + target_batch_size
            if self.labeled_data:
                per_dataset_batch += labeled_batch_size

            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = ResettableDataLoader(
                dataset=combined_dataset,
                batch_size=per_dataset_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=self.trainset.collate_fn,
                drop_last=True,
            )
            LOGGER.info(f"DataLoader batch sizes: source={source_batch_size}, target={target_batch_size}, labeled={labeled_batch_size}, total={per_dataset_batch}")
            sys.stdout.flush()
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def compute_yolo_loss(self, batch):
        """Compute YOLO detection loss for a batch."""
        imgs, targets, _, _ = batch
        imgs = imgs.to(self.device, non_blocking=True)
        targets = targets.to(self.device, non_blocking=True)

        # Forward pass
        preds = self.model(imgs)

        # Initialize loss function
        loss_fn = v8DetectionLoss(self.model)  # No nc parameter

        # Compute loss
        loss, loss_items = loss_fn(preds, (imgs, targets))  # Pass batch as tuple

        return loss, loss_items

    def forward_batch(self, batch, epoch=0):
        LOGGER.debug(f"Entering forward_batch for epoch {epoch}, batch {self.batch_count + 1}")
        sys.stdout.flush()

        self.current_epoch = epoch
        self.update_consistency_weight(epoch)
        self.batch_count += 1

        LOGGER.debug(f"Batch {self.batch_count}, Epoch {self.current_epoch}, Total Batch Size: {batch[0].shape[0]}")
        sys.stdout.flush()

        if self.labeled_data:
            bs = batch[0].shape[0] // 3
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:2*bs] for b in batch]
            labeled_batch = [b[2*bs:] for b in batch]
            LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}, Labeled Batch Size: {labeled_batch[0].shape[0]}")
        else:
            bs = batch[0].shape[0] // 2
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:] for b in batch]
            labeled_batch = None
            LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}")
        sys.stdout.flush()

        # Compute supervised loss on source domain
        loss, loss_items = self.compute_yolo_loss(source_batch)
        supervised_loss = loss.item()

        # Compute features for domain adaptation
        with torch.no_grad():
            self.model.eval()
            _, source_feats = self.model(source_batch[0], return_feats=True)
            _, target_feats = self.model(target_batch[0], return_feats=True)
            self.model.train()

        # Consistency loss
        consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
        loss += self.consistency_weight * consistency_loss

        # Supervised loss on labeled target data
        labeled_loss = 0.0
        if labeled_batch:
            labeled_loss, _ = self.compute_yolo_loss(labeled_batch)
            loss += 2.0 * labeled_loss
            labeled_loss = labeled_loss.item()

        # Domain adversarial loss
        domain_preds = self.discriminator(source_feats + target_feats)
        domain_labels = torch.cat([
            torch.zeros(len(source_feats[0]), device=self.device),
            torch.ones(len(target_feats[0]), device=self.device)
        ])
        adv_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        loss += self.adversarial_weight * adv_loss

        # Update discriminator
        self.discriminator.optim.zero_grad()
        adv_loss.backward(retain_graph=True)
        self.discriminator.optim.step()

        # Compute domain accuracy
        domain_acc = ((domain_preds > 0).float() == domain_labels).float().mean().item()

        # Log metrics
        metrics = {
            'epoch': epoch,
            'total_loss': loss.item(),
            'supervised_loss': supervised_loss,
            'consistency_loss': consistency_loss.item(),
            'adversarial_loss': adv_loss.item(),
            'labeled_loss': labeled_loss,
            'domain_accuracy': domain_acc,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'box_loss': loss_items[0],
            'cls_loss': loss_items[1],
            'dfl_loss': loss_items[2],
        }
        self.training_metrics.append(metrics)
        self.domain_metrics.append({
            'epoch': epoch,
            'domain_accuracy': domain_acc,
            'adv_loss': adv_loss.item()
        })

        # Print metrics every batch with LOGGER.info and print
        log_message = (
            f"\n{'-'*40}\n"
            f"Epoch {self.current_epoch}, Batch {self.batch_count} - Batch Metrics\n"
            f"{'-'*40}\n"
            f"{'Metric':<20} {'Value':>10}\n"
            f"{'-'*30}\n"
            f"{'Total Loss':<20} {loss.item():>10.4f}\n"
            f"{'Supervised Loss':<20} {supervised_loss:>10.4f}\n"
            f"{'Consistency Loss':<20} {consistency_loss.item():>10.4f}\n"
            f"{'Adversarial Loss':<20} {adv_loss.item():>10.4f}\n"
            f"{'Labeled Loss':<20} {labeled_loss:>10.4f}\n"
            f"{'Domain Accuracy':<20} {domain_acc:>10.4f}\n"
            f"{'Box Loss':<20} {loss_items[0]:>10.4f}\n"
            f"{'Cls Loss':<20} {loss_items[1]:>10.4f}\n"
            f"{'Dfl Loss':<20} {loss_items[2]:>10.4f}\n"
            f"{'-'*40}"
        )
        LOGGER.info(log_message)
        print(log_message)
        sys.stdout.flush()

        loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
        return loss, loss_items

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.05 * epoch)

    def save_metrics(self, metrics=None):
        super().save_metrics(metrics=metrics)

        if RANK in (-1, 0):
            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)

            # Combine Ultralytics metrics with custom metrics
            combined_metrics = []
            if metrics:
                ultralytics_metrics = {
                    'epoch': self.current_epoch,
                    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                for key, value in metrics.items():
                    if key.startswith('metrics/') or key in ['box_loss', 'cls_loss', 'dfl_loss', 'lr0', 'lr1', 'lr2']:
                        ultralytics_metrics[key] = value
                combined_metrics.append(ultralytics_metrics)

            # Append custom metrics from self.training_metrics
            for custom_metric in self.training_metrics:
                if custom_metric['epoch'] == self.current_epoch:
                    combined_metrics.append(custom_metric)

            if combined_metrics:
                metrics_df = pd.DataFrame(combined_metrics)
                metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                metrics_df.to_csv(metrics_path, index=False)
                LOGGER.info(f'Saved training metrics to {metrics_path}')
                sys.stdout.flush()

            if self.domain_metrics:
                domain_df = pd.DataFrame(self.domain_metrics)
                domain_path = os.path.join(metrics_dir, 'domain_metrics.csv')
                domain_df.to_csv(domain_path, index=False)
                LOGGER.info(f'Saved domain adaptation metrics to {domain_path}')
                sys.stdout.flush()

            if self.thermal_metrics:
                thermal_df = pd.DataFrame(self.thermal_metrics)
                thermal_path = os.path.join(metrics_dir, 'validation_metrics.csv')
                thermal_df.to_csv(thermal_path, index=False)
                LOGGER.info(f'Saved validation metrics to {thermal_path}')
                sys.stdout.flush()

    def on_train_epoch_end(self):
        if self.current_epoch % 20 == 0 and self.current_epoch > 0:
            LOGGER.info(f"\n{'='*50}")
            LOGGER.info(f"Validation Triggered at Epoch {self.current_epoch}")
            LOGGER.info(f"{'='*50}")
            sys.stdout.flush()
            try:
                LOGGER.info("Validating Source Domain (Visible)...")
                sys.stdout.flush()
                val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                LOGGER.info("Validating Target Domain (Thermal)...")
                sys.stdout.flush()
                thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

                LOGGER.info(f"\n{'-'*50}")
                LOGGER.info(f"Validation Results at Epoch {self.current_epoch}")
                LOGGER.info(f"{'-'*50}")
                LOGGER.info(f"Source Domain (Visible):")
                LOGGER.info(f"  mAP50: {val_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {val_results.box.map:.4f}")
                LOGGER.info(f"Target Domain (Thermal):")
                LOGGER.info(f"  mAP50: {thermal_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {thermal_results.box.map:.4f}")
                LOGGER.info(f"{'='*50}")
                sys.stdout.flush()

                self.thermal_metrics.append({
                    'epoch': self.current_epoch,
                    'source_mAP50': val_results.box.map50,
                    'source_mAP50-95': val_results.box.map,
                    'thermal_mAP50': thermal_results.box.map50,
                    'thermal_mAP50-95': thermal_results.box.map,
                })
            except Exception as e:
                LOGGER.error(f"Validation failed at epoch {self.current_epoch}: {e}")
                sys.stdout.flush()
        self.batch_count = 0

    def on_train_end(self):
        self.save_metrics()

        if RANK in (-1, 0):
            val_results = self.model.val(data=self.args.data, batch=self.args.batch)
            thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

            LOGGER.info(f"\nFinal Validation Results:")
            LOGGER.info(f"Source Domain (Visible): mAP50={val_results.box.map50:.4f}, mAP50-95={val_results.box.map:.4f}")
            LOGGER.info(f"Target Domain (Thermal): mAP50={thermal_results.box.map50:.4f}, mAP50-95={thermal_results.box.map:.4f}")
            sys.stdout.flush()

            # Print average losses per epoch
            metrics_path = os.path.join(self.save_dir, 'metrics', 'training_metrics.csv')
            if os.path.isfile(metrics_path):
                metrics_df = pd.read_csv(metrics_path)
                avg_metrics = metrics_df.groupby('epoch').mean(numeric_only=True)
                LOGGER.info(f"\n{'='*50}")
                LOGGER.info("Average Losses Per Epoch")
                LOGGER.info(f"{'-'*50}")
                for epoch in avg_metrics.index:
                    LOGGER.info(f"Epoch {int(epoch)}:")
                    LOGGER.info(f"  Total Loss: {avg_metrics.loc[epoch, 'total_loss']:.4f}")
                    LOGGER.info(f"  Supervised Loss: {avg_metrics.loc[epoch, 'supervised_loss']:.4f}")
                    LOGGER.info(f"  Consistency Loss: {avg_metrics.loc[epoch, 'consistency_loss']:.4f}")
                    LOGGER.info(f"  Adversarial Loss: {avg_metrics.loc[epoch, 'adversarial_loss']:.4f}")
                    LOGGER.info(f"  Labeled Loss: {avg_metrics.loc[epoch, 'labeled_loss']:.4f}")
                    LOGGER.info(f"  Domain Accuracy: {avg_metrics.loc[epoch, 'domain_accuracy']:.4f}")
                LOGGER.info(f"{'='*50}")
                sys.stdout.flush()

            duration = datetime.now() - self.start_time
            LOGGER.info(f"\nTraining completed in {duration}")
            sys.stdout.flush()

        return super().on_train_end()

class CustomTrainerFactory:
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg):
        self.target_domain_data_cfg = target_domain_data_cfg
        self.labeled_thermal_data_cfg = labeled_thermal_data_cfg

    def __call__(self, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        return CustomTrainer(
            target_domain_data_cfg=self.target_domain_data_cfg,
            labeled_thermal_data_cfg=self.labeled_thermal_data_cfg,
            overrides=overrides,
            _callbacks=_callbacks
        )

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)

    kwargs = dict(
        imgsz=640,
        epochs=50,
        val=True,
        workers=4,
        batch=16,
        seed=9527,
        patience=50,
        save=True,
        plots=True,
        lr0=0.002,
        lrf=0.002,
        warmup_epochs=5,
        optimizer='AdamW',
        weight_decay=0.0005,
        momentum=0.9,
        data='data/visible.yaml'
    )

    exp_name = 'visible_to_thermal_semisupervised'
    base_dir = '/content/pguard/domain_adaptation_process'
    exp_dir = os.path.join(base_dir, exp_name)

    os.makedirs(base_dir, exist_ok=True)
    if os.path.isdir(exp_dir):
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)

    # Setup file logging
    setup_file_logging(exp_dir)

    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    trainer = CustomTrainerFactory(
        target_domain_data_cfg=data_thermal,
        labeled_thermal_data_cfg=data_thermal_lbl
    )

    results = model.train(
        trainer=trainer,
        name=exp_dir,
        exist_ok=True,
        **kwargs
    )

    trainer_instance = results.trainer
    if trainer_instance.training_metrics:
        metrics_df = pd.DataFrame(trainer_instance.training_metrics)
        metrics_csv_path = os.path.join(exp_dir, 'training_metrics.csv')
        metrics_df.to_csv(metrics_csv_path, index=False)
        LOGGER.info(f'Saved training metrics to {metrics_csv_path}')
        sys.stdout.flush()

    val_dir = os.path.join(base_dir, 'val_thermal')
    os.makedirs(val_dir, exist_ok=True)
    val_results = model.val(data=data_thermal, name=val_dir, batch=8)

    LOGGER.info("\nTraining completed successfully!")
    LOGGER.info(f"Results saved to: {exp_dir}")
    LOGGER.info(f"Validation results saved to: {val_dir}")
    sys.stdout.flush()

if __name__ == '__main__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
%%writefile /content/drive/MyDrive/Domain-adaptation-object-detection-with-YOLOv8/data/thermal.yaml
path: /content/drive/My Drive/pguard/thermal
train: train/images
val: val/images
nc: 1
names: ['lp']
augment:
  gaussian_noise: 0.2
  blur: 5.0
  mosaic: 0.7
  mixup: 0.3
  hsv_v: 0.5
  degrees: 15.0
  translate: 0.2
  scale: 0.7
  shear: 5.0
  fliplr: 0.5
  hsv_h: 0.0
  hsv_s: 0.0

Overwriting /content/drive/MyDrive/Domain-adaptation-object-detection-with-YOLOv8/data/thermal.yaml


In [ ]:
import shutil
import os

directory = '/content/pguard/domain_adaptation_process'
if os.path.exists(directory):
    shutil.rmtree(directory)
    print(f"Successfully deleted directory: {directory}")
else:
    print(f"Directory does not exist: {directory}")

if not os.path.exists(directory):
    print(f"Confirmed: {directory} has been removed.")
else:
    print(f"Error: {directory} still exists.")

Successfully deleted directory: /content/pguard/domain_adaptation_process
Confirmed: /content/pguard/domain_adaptation_process has been removed.


In [ ]:
!rmdir  /content/pguard/domain_adaptation_process

rmdir: failed to remove '/content/pguard/domain_adaptation_process': No such file or directory


In [ ]:
import os

def clear_dataset_caches(dataset_dirs):
    for dir_path in dataset_dirs:
        cache_path = os.path.join(dir_path, 'labels.cache')
        if os.path.exists(cache_path):
            os.remove(cache_path)
            print(f"Deleted cache: {cache_path}")
        else:
            print(f"No cache found: {cache_path}")

dataset_dirs = [
    '/content/drive/My Drive/pguard/thermal/train',
    '/content/drive/My Drive/pguard/thermal/val',
    '/content/drive/My Drive/pguard/train',
    '/content/drive/My Drive/pguard/val',
    '/content/drive/My Drive/pguard/thermal_labeled/train',
    '/content/drive/My Drive/pguard/thermal_labeled/val',
]

clear_dataset_caches(dataset_dirs)

Deleted cache: /content/drive/My Drive/pguard/thermal/train/labels.cache
Deleted cache: /content/drive/My Drive/pguard/thermal/val/labels.cache
Deleted cache: /content/drive/My Drive/pguard/train/labels.cache
Deleted cache: /content/drive/My Drive/pguard/val/labels.cache
No cache found: /content/drive/My Drive/pguard/thermal_labeled/train/labels.cache
No cache found: /content/drive/My Drive/pguard/thermal_labeled/val/labels.cache


In [ ]:
%%writefile /content/Domain-adaptation-object-detection-with-YOLOv8/train.py
import os
import shutil
import pandas as pd
from datetime import datetime
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from torch.utils.data import Dataset, DataLoader
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import RANK, TQDM, colorstr, emojis, clean_url, LOGGER
from ultralytics import YOLO
import sys
import logging
from ultralytics.nn.tasks import v8DetectionLoss

# Set script directory for consistent relative paths
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

# Ensure LOGGER is set to INFO level for visibility
logging.getLogger('ultralytics').setLevel(logging.INFO)
LOGGER.setLevel(logging.INFO)

# Add file handler for logging
def setup_file_logging(log_dir):
    os.makedirs(log_dir, exist_ok=True)
    file_handler = logging.FileHandler(os.path.join(log_dir, 'training_log.txt'))
    file_handler.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    LOGGER.addHandler(file_handler)

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, 'weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h//2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(nn.Linear(dim_h//2, dim_h//2), nn.LeakyReLU(0.2, inplace=True))
            for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h//2*4, dim_h//2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),  # Added dropout
            nn.Linear(dim_h//2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        part, rest = x.split(x.shape[1]//2, dim=1)
        xs = [part]
        for m in self.neck:
            rest = m(rest)
            xs.append(rest)
        return self.head(torch.cat(xs, dim=1))

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        chs = chs or [64,128,256]
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i==0 else chs[i]*2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64), nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i+1] if i+1 < len(chs) else chs[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(chs[i+1] if i+1 < len(chs) else chs[i]), nn.SiLU(inplace=True)
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, feats):
        with torch.cuda.amp.autocast(self.amp):
            out = None
            for f, lay in zip(feats, self.layers):
                x = lay(f)
                out = x if out is None else torch.cat((out, x), dim=1)
            return self.head(self.grl(out))

class CombinedDataset(Dataset):
    def __init__(self, source_dataset, target_dataset, labeled_dataset=None):
        self.source_dataset = source_dataset
        self.target_dataset = target_dataset
        self.labeled_dataset = labeled_dataset
        self.datasets = [source_dataset, target_dataset]
        if labeled_dataset is not None:
            self.datasets.append(labeled_dataset)
        self.lengths = [len(ds) for ds in self.datasets]
        self.min_length = min(self.lengths)
        self.total_length = self.min_length * len(self.datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):
        dataset_idx = idx % len(self.datasets)
        sample_idx = idx // len(self.datasets)
        return self.datasets[dataset_idx][sample_idx % self.lengths[dataset_idx]]

    @property
    def labels(self):
        labels = []
        for ds in [self.source_dataset, self.labeled_dataset]:
            if ds is not None and hasattr(ds, 'labels'):
                labels.extend(ds.labels)
        return labels

class ResettableDataLoader(DataLoader):
    def reset(self):
        # Reset the iterator (simulating Ultralytics' InfiniteDataLoader behavior)
        self._iterator = iter(self)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        LOGGER.info(f"Initializing CustomTrainer with data: {self.args.data}")
        sys.stdout.flush()
        self.t_data = check_det_dataset(target_domain_data_cfg)
        self.t_train, self.t_val = self.get_dataset(data=self.t_data, require_labels=False)
        self.labeled_data = None
        if labeled_thermal_data_cfg:
            self.labeled_data = check_det_dataset(labeled_thermal_data_cfg)
            self.labeled_train, _ = self.get_dataset(data=self.labeled_data, require_labels=True)
            LOGGER.info(f"Labeled thermal training dataset size: {len(self.labeled_train)} images (path: {self.labeled_data['train']})")
        self.additional_models, self.thermal_metrics = [], []
        self.training_metrics = []  # For batch-level metrics
        self.domain_metrics = []   # For domain adaptation metrics
        self.add_callback('on_train_start', self._init_discriminator)
        self.consistency_weight = 1.0
        self.adversarial_weight = 0.5
        self.current_epoch = 0
        self.start_time = datetime.now()
        self.batch_count = 0

        # Adjust training parameters
        self.args.patience = 50
        self.args.lr0 = 0.001
        self.args.lrf = 0.001
        self.args.warmup_epochs = 5

    def _init_discriminator(self, *args, **kwargs):
        self.discriminator = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator)

    def get_dataset(self, data=None, require_labels=True):
        if data is None:
            return super().get_dataset()
        from ultralytics.data.dataset import YOLODataset
        return (
            YOLODataset(img_path=data['train'], imgsz=self.args.imgsz, data=data, augment=True, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls),
            YOLODataset(img_path=data['val'], imgsz=self.args.imgsz, data=data, augment=False, cache=self.args.cache, prefix='', rect=False, single_cls=self.args.single_cls)
        )

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            # Ensure self.trainset is a dataset object
            source_data = check_det_dataset(self.args.data)
            self.trainset, _ = self.get_dataset(data=source_data, require_labels=True)

            total_batch = self.args.batch
            source_batch_size = total_batch // 2
            target_batch_size = total_batch // 2
            labeled_batch_size = total_batch // 4 if self.labeled_data else 0

            combined_dataset = CombinedDataset(
                source_dataset=self.trainset,
                target_dataset=self.t_train,
                labeled_dataset=self.labeled_train if self.labeled_data else None
            )

            per_dataset_batch = source_batch_size + target_batch_size
            if self.labeled_data:
                per_dataset_batch += labeled_batch_size

            sampler = None if rank == -1 else torch.utils.data.distributed.DistributedSampler(combined_dataset, shuffle=True)
            loader = ResettableDataLoader(
                dataset=combined_dataset,
                batch_size=per_dataset_batch,
                shuffle=sampler is None,
                num_workers=self.args.workers,
                sampler=sampler,
                pin_memory=True,
                collate_fn=self.trainset.collate_fn,
                drop_last=True,
            )
            LOGGER.info(f"DataLoader batch sizes: source={source_batch_size}, target={target_batch_size}, labeled={labeled_batch_size}, total={per_dataset_batch}")
            sys.stdout.flush()
            return loader
        return super().get_dataloader(dataset, batch_size=batch_size, rank=rank, mode=mode)

    def compute_yolo_loss(self, batch):
        """Compute YOLO detection loss for a batch."""
        imgs, targets, _, _ = batch
        imgs = imgs.to(self.device, non_blocking=True)
        targets = targets.to(self.device, non_blocking=True)

        # Forward pass
        preds = self.model(imgs)

        # Initialize loss function
        loss_fn = v8DetectionLoss(self.model)

        # Compute loss
        loss, loss_items = loss_fn(preds, (imgs, targets))

        return loss, loss_items

    def forward_batch(self, batch, epoch=0):
        LOGGER.debug(f"Entering forward_batch for epoch {epoch}, batch {self.batch_count + 1}")
        sys.stdout.flush()

        self.current_epoch = epoch
        self.update_consistency_weight(epoch)
        self.batch_count += 1

        LOGGER.debug(f"Batch {self.batch_count}, Epoch {self.current_epoch}, Total Batch Size: {batch[0].shape[0]}")
        sys.stdout.flush()

        if self.labeled_data:
            bs = batch[0].shape[0] // 3
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:2*bs] for b in batch]
            labeled_batch = [b[2*bs:] for b in batch]
            LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}, Labeled Batch Size: {labeled_batch[0].shape[0]}")
        else:
            bs = batch[0].shape[0] // 2
            source_batch = [b[:bs] for b in batch]
            target_batch = [b[bs:] for b in batch]
            labeled_batch = None
            LOGGER.debug(f"Source Batch Size: {source_batch[0].shape[0]}, Target Batch Size: {target_batch[0].shape[0]}")
        sys.stdout.flush()

        # Compute supervised loss on source domain
        loss, loss_items = self.compute_yolo_loss(source_batch)
        supervised_loss = loss.item()

        # Compute features for domain adaptation
        with torch.no_grad():
            self.model.eval()
            _, source_feats = self.model(source_batch[0], return_feats=True)
            _, target_feats = self.model(target_batch[0], return_feats=True)
            self.model.train()

        # Consistency loss
        consistency_loss = self.compute_consistency_loss(source_feats, target_feats)
        loss += self.consistency_weight * consistency_loss

        # Supervised loss on labeled target data
        labeled_loss = 0.0
        if labeled_batch:
            labeled_loss, _ = self.compute_yolo_loss(labeled_batch)
            loss += 0.05 * labeled_loss  # Further reduced to 0.05
            labeled_loss = labeled_loss.item()

        # Domain adversarial loss
        domain_preds = self.discriminator(source_feats + target_feats)
        domain_labels = torch.cat([
            torch.zeros(len(source_feats[0]), device=self.device),
            torch.ones(len(target_feats[0]), device=self.device)
        ])
        adv_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        loss += 3.0 * adv_loss  # Further increased to 3.0

        # Update discriminator
        self.discriminator.optim.zero_grad()
        adv_loss.backward(retain_graph=True)
        self.discriminator.optim.step()

        # Compute domain accuracy
        domain_acc = ((domain_preds > 0).float() == domain_labels).float().mean().item()

        # Log metrics
        metrics = {
            'epoch': epoch,
            'total_loss': loss.item(),
            'supervised_loss': supervised_loss,
            'consistency_loss': consistency_loss.item(),
            'adversarial_loss': adv_loss.item(),
            'labeled_loss': labeled_loss,
            'domain_accuracy': domain_acc,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'box_loss': loss_items[0],
            'cls_loss': loss_items[1],
            'dfl_loss': loss_items[2],
        }
        self.training_metrics.append(metrics)
        self.domain_metrics.append({
            'epoch': epoch,
            'domain_accuracy': domain_acc,
            'adv_loss': adv_loss.item()
        })

        # Print metrics every batch with LOGGER.info and print
        log_message = (
            f"\n{'-'*40}\n"
            f"Epoch {self.current_epoch}, Batch {self.batch_count} - Batch Metrics\n"
            f"{'-'*40}\n"
            f"{'Metric':<20} {'Value':>10}\n"
            f"{'-'*30}\n"
            f"{'Total Loss':<20} {loss.item():>10.4f}\n"
            f"{'Supervised Loss':<20} {supervised_loss:>10.4f}\n"
            f"{'Consistency Loss':<20} {consistency_loss.item():>10.4f}\n"
            f"{'Adversarial Loss':<20} {adv_loss.item():>10.4f}\n"
            f"{'Labeled Loss':<20} {labeled_loss:>10.4f}\n"
            f"{'Domain Accuracy':<20} {domain_acc:>10.4f}\n"
            f"{'Box Loss':<20} {loss_items[0]:>10.4f}\n"
            f"{'Cls Loss':<20} {loss_items[1]:>10.4f}\n"
            f"{'Dfl Loss':<20} {loss_items[2]:>10.4f}\n"
            f"{'-'*40}"
        )
        LOGGER.info(log_message)
        print(log_message)
        sys.stdout.flush()

        loss_items = list(loss_items) + [consistency_loss.item(), adv_loss.item()]
        return loss, loss_items

    def compute_consistency_loss(self, source_feats, target_feats):
        loss = 0
        for s_feat, t_feat in zip(source_feats, target_feats):
            s_feat = F.normalize(s_feat, dim=1)
            t_feat = F.normalize(t_feat, dim=1)
            loss += F.mse_loss(s_feat, t_feat)
        return loss / len(source_feats)

    def update_consistency_weight(self, epoch):
        self.consistency_weight = min(1.0, 0.05 * epoch)

    def save_metrics(self, metrics=None):
        super().save_metrics(metrics=metrics)

        if RANK in (-1, 0):
            metrics_dir = os.path.join(self.save_dir, 'metrics')
            os.makedirs(metrics_dir, exist_ok=True)

            # Combine Ultralytics metrics with custom metrics
            combined_metrics = []
            if metrics:
                ultralytics_metrics = {
                    'epoch': self.current_epoch,
                    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                for key, value in metrics.items():
                    if key.startswith('metrics/') or key in ['box_loss', 'cls_loss', 'dfl_loss', 'lr0', 'lr1', 'lr2']:
                        ultralytics_metrics[key] = value
                combined_metrics.append(ultralytics_metrics)

            # Append custom metrics from self.training_metrics
            for custom_metric in self.training_metrics:
                if custom_metric['epoch'] == self.current_epoch:
                    combined_metrics.append(custom_metric)

            if combined_metrics:
                metrics_df = pd.DataFrame(combined_metrics)
                metrics_path = os.path.join(metrics_dir, 'training_metrics.csv')
                metrics_df.to_csv(metrics_path, index=False)
                LOGGER.info(f'Saved training metrics to {metrics_path}')
                sys.stdout.flush()

            if self.domain_metrics:
                domain_df = pd.DataFrame(self.domain_metrics)
                domain_path = os.path.join(metrics_dir, 'domain_metrics.csv')
                domain_df.to_csv(domain_path, index=False)
                LOGGER.info(f'Saved domain adaptation metrics to {domain_path}')
                sys.stdout.flush()

            if self.thermal_metrics:
                thermal_df = pd.DataFrame(self.thermal_metrics)
                thermal_path = os.path.join(metrics_dir, 'validation_metrics.csv')
                thermal_df.to_csv(thermal_path, index=False)
                LOGGER.info(f'Saved validation metrics to {thermal_path}')
                sys.stdout.flush()

    def on_train_epoch_end(self):
        # Log domain accuracy summary
        if self.domain_metrics:
            epoch_metrics = [m for m in self.domain_metrics if m['epoch'] == self.current_epoch]
            if epoch_metrics:
                avg_domain_acc = sum(m['domain_accuracy'] for m in epoch_metrics) / len(epoch_metrics)
                avg_adv_loss = sum(m['adv_loss'] for m in epoch_metrics) / len(epoch_metrics)
                LOGGER.info(f"\nEpoch {self.current_epoch} Domain Metrics:")
                LOGGER.info(f"  Average Domain Accuracy: {avg_domain_acc:.4f}")
                LOGGER.info(f"  Average Adversarial Loss: {avg_adv_loss:.4f}")
                sys.stdout.flush()

        if self.current_epoch % 5 == 0 and self.current_epoch > 0:
            LOGGER.info(f"\n{'='*50}")
            LOGGER.info(f"Validation Triggered at Epoch {self.current_epoch}")
            LOGGER.info(f"{'='*50}")
            sys.stdout.flush()
            try:
                LOGGER.info("Validating Source Domain (Visible)...")
                sys.stdout.flush()
                val_results = self.model.val(data=self.args.data, batch=self.args.batch)
                LOGGER.info("Validating Target Domain (Thermal)...")
                sys.stdout.flush()
                thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

                LOGGER.info(f"\n{'-'*50}")
                LOGGER.info(f"Validation Results at Epoch {self.current_epoch}")
                LOGGER.info(f"{'-'*50}")
                LOGGER.info(f"Source Domain (Visible):")
                LOGGER.info(f"  mAP50: {val_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {val_results.box.map:.4f}")
                LOGGER.info(f"Target Domain (Thermal):")
                LOGGER.info(f"  mAP50: {thermal_results.box.map50:.4f}")
                LOGGER.info(f"  mAP50-95: {thermal_results.box.map:.4f}")
                LOGGER.info(f"{'='*50}")
                sys.stdout.flush()

                self.thermal_metrics.append({
                    'epoch': self.current_epoch,
                    'source_mAP50': val_results.box.map50,
                    'source_mAP50-95': val_results.box.map,
                    'thermal_mAP50': thermal_results.box.map50,
                    'thermal_mAP50-95': thermal_results.box.map,
                })

                # Early stopping based on thermal mAP50
                if thermal_results.box.map50 <= 0.95:
                    LOGGER.info(f"Early stopping triggered: Thermal mAP50 ({thermal_results.box.map50:.4f}) <= 0.95.")
                    sys.stdout.flush()
                    raise StopIteration("Early stopping due to thermal mAP50 target reached.")
            except StopIteration as e:
                raise e
            except Exception as e:
                LOGGER.error(f"Validation failed at epoch {self.current_epoch}: {e}")
                sys.stdout.flush()
        self.batch_count = 0

    def on_train_end(self):
        self.save_metrics()

        if RANK in (-1, 0):
            val_results = self.model.val(data=self.args.data, batch=self.args.batch)
            thermal_results = self.model.val(data=self.t_data, batch=self.args.batch)

            LOGGER.info(f"\nFinal Validation Results:")
            LOGGER.info(f"Source Domain (Visible): mAP50={val_results.box.map50:.4f}, mAP50-95={val_results.box.map:.4f}")
            LOGGER.info(f"Target Domain (Thermal): mAP50={thermal_results.box.map50:.4f}, mAP50-95={thermal_results.box.map:.4f}")
            sys.stdout.flush()

            # Print average losses per epoch
            metrics_path = os.path.join(self.save_dir, 'metrics', 'training_metrics.csv')
            if os.path.isfile(metrics_path):
                metrics_df = pd.read_csv(metrics_path)
                avg_metrics = metrics_df.groupby('epoch').mean(numeric_only=True)
                LOGGER.info(f"\n{'='*50}")
                LOGGER.info("Average Losses Per Epoch")
                LOGGER.info(f"{'-'*50}")
                for epoch in avg_metrics.index:
                    LOGGER.info(f"Epoch {int(epoch)}:")
                    LOGGER.info(f"  Total Loss: {avg_metrics.loc[epoch, 'total_loss']:.4f}")
                    LOGGER.info(f"  Supervised Loss: {avg_metrics.loc[epoch, 'supervised_loss']:.4f}")
                    LOGGER.info(f"  Consistency Loss: {avg_metrics.loc[epoch, 'consistency_loss']:.4f}")
                    LOGGER.info(f"  Adversarial Loss: {avg_metrics.loc[epoch, 'adversarial_loss']:.4f}")
                    LOGGER.info(f"  Labeled Loss: {avg_metrics.loc[epoch, 'labeled_loss']:.4f}")
                    LOGGER.info(f"  Domain Accuracy: {avg_metrics.loc[epoch, 'domain_accuracy']:.4f}")
                LOGGER.info(f"{'='*50}")
                sys.stdout.flush()

            duration = datetime.now() - self.start_time
            LOGGER.info(f"\nTraining completed in {duration}")
            sys.stdout.flush()

        return super().on_train_end()

class CustomTrainerFactory:
    def __init__(self, target_domain_data_cfg, labeled_thermal_data_cfg):
        self.target_domain_data_cfg = target_domain_data_cfg
        self.labeled_thermal_data_cfg = labeled_thermal_data_cfg

    def __call__(self, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        return CustomTrainer(
            target_domain_data_cfg=self.target_domain_data_cfg,
            labeled_thermal_data_cfg=self.labeled_thermal_data_cfg,
            overrides=overrides,
            _callbacks=_callbacks
        )

def resolve_yaml(rel_path):
    p1 = os.path.join(SCRIPT_DIR, rel_path)
    p2 = os.path.join(os.getcwd(), rel_path)
    for p in (p1, p2):
        if os.path.isfile(p): return p
    raise FileNotFoundError(f"YAML file not found: {rel_path}")

def main():
    seed_everything(9527)

    kwargs = dict(
        imgsz=640,
        epochs=30,  # Reduced from 40 to 30
        val=True,
        workers=4,
        batch=16,
        seed=9527,
        patience=50,
        save=True,
        plots=True,
        lr0=0.001,
        lrf=0.001,
        warmup_epochs=5,
        optimizer='AdamW',
        weight_decay=0.0005,
        momentum=0.9,
        data='data/visible.yaml'
    )

    exp_name = 'visible_to_thermal_semisupervised'
    base_dir = '/content/pguard/domain_adaptation_process'
    exp_dir = os.path.join(base_dir, exp_name)

    os.makedirs(base_dir, exist_ok=True)
    if os.path.isdir(exp_dir):
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)

    # Setup file logging
    setup_file_logging(exp_dir)

    w_local = os.path.join(SCRIPT_DIR, 'runs', 'train', 'yolov8s_exp', 'weights', 'best.pt')
    w_drive = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    model_path = w_local if os.path.isfile(w_local) else w_drive
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"best.pt not found at {w_local} or {w_drive}")
    model = YOLO(model_path)

    data_visible = resolve_yaml('data/visible.yaml')
    data_thermal = resolve_yaml('data/thermal.yaml')
    data_thermal_lbl = resolve_yaml('data/thermal_labeled.yaml')

    trainer = CustomTrainerFactory(
        target_domain_data_cfg=data_thermal,
        labeled_thermal_data_cfg=data_thermal_lbl
    )

    results = model.train(
        trainer=trainer,
        name=exp_dir,
        exist_ok=True,
        **kwargs
    )

    trainer_instance = results.trainer
    if trainer_instance.training_metrics:
        metrics_df = pd.DataFrame(trainer_instance.training_metrics)
        metrics_csv_path = os.path.join(exp_dir, 'training_metrics.csv')
        metrics_df.to_csv(metrics_csv_path, index=False)
        LOGGER.info(f'Saved training metrics to {metrics_csv_path}')
        sys.stdout.flush()

    val_dir = os.path.join(base_dir, 'val_thermal')
    os.makedirs(val_dir, exist_ok=True)
    val_results = model.val(data=data_thermal, name=val_dir, batch=8)

    LOGGER.info("\nTraining completed successfully!")
    LOGGER.info(f"Results saved to: {exp_dir}")
    LOGGER.info(f"Validation results saved to: {val_dir}")
    sys.stdout.flush()

if __name__ == '__main__':
    main()

Overwriting /content/Domain-adaptation-object-detection-with-YOLOv8/train.py


In [ ]:
%cd /content/Domain-adaptation-object-detection-with-YOLOv8
!python train.py

/content/Domain-adaptation-object-detection-with-YOLOv8
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.0.1+cu117 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/visible.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt, momentum=0.9, mosaic=1.0, multi_scale=False, name=visible_to_thermal_semisupervised, nbs=64, nms=False,

In [ ]:
import shutil
import os

# Source and destination paths
source_dir = '/content/pguard/domain_adaptation_process'
dest_dir = '/content/drive/My Drive/pguard/domain_adaptation_process1'

# Create destination directory if it doesn't exist
os.makedirs(os.path.dirname(dest_dir), exist_ok=True)

# Copy the entire folder
try:
    shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
    print(f"Successfully copied {source_dir} to {dest_dir}")
except Exception as e:
    print(f"Error copying folder: {e}")

Successfully copied /content/pguard/domain_adaptation_process to /content/drive/My Drive/pguard/domain_adaptation_process1


In [ ]:
from ultralytics import YOLO
import os
import numpy as np

# Define paths
model_path = '/content/drive/MyDrive/pguard/domain_adaptation_process1/visible_to_thermal_semisupervised/weights/best.pt'
thermal_yaml = '/content/Domain-adaptation-object-detection-with-YOLOv8/data/thermal.yaml'
visible_yaml = '/content/Domain-adaptation-object-detection-with-YOLOv8/data/visible.yaml'

# Verify that files exist
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found at {model_path}")
if not os.path.exists(thermal_yaml):
    raise FileNotFoundError(f"Thermal YAML file not found at {thermal_yaml}")
if not os.path.exists(visible_yaml):
    raise FileNotFoundError(f"Visible YAML file not found at {visible_yaml}")

# Load the model
model = YOLO(model_path)

# Function to print validation results
def print_results(results, domain_name, num_images, num_instances):
    print(f"\n{domain_name} Domain Validation Results:")
    print(f"mAP50: {results.box.map50:.4f}")
    print(f"mAP50-95: {results.box.map:.4f}")
    # Handle Precision and Recall as arrays or scalars
    precision = results.box.p[0] if isinstance(results.box.p, (list, np.ndarray)) else results.box.p
    recall = results.box.r[0] if isinstance(results.box.r, (list, np.ndarray)) else results.box.r
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Images: {num_images}")
    print(f"Instances: {num_instances}")

# Validate on thermal dataset
print("\nValidating on Thermal Dataset...")
try:
    thermal_results = model.val(data=thermal_yaml, batch=8, device='0')  # Use GPU (CUDA:0)
    print_results(thermal_results, "Thermal", num_images= 526, num_instances=526)
except Exception as e:
    print(f"Error during thermal validation: {e}")

# Validate on visible dataset
print("\nValidating on Visible Dataset...")
try:
    visible_results = model.val(data=visible_yaml, batch=8, device='0')  # Use GPU (CUDA:0)
    print_results(visible_results, "Visible", num_images=638, num_instances=956)
except Exception as e:
    print(f"Error during visible validation: {e}")


Validating on Thermal Dataset...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 214.2±64.9 MB/s, size: 408.9 KB)


val: Scanning /content/drive/My Drive/pguard/thermal/val/labels.cache... 526 images, 0 backgrounds, 0 corrupt: 100%|██████████| 526/526 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 66/66 [00:05<00:00, 11.13it/s]


                   all        526        526      0.944      0.897      0.971      0.721
Speed: 0.3ms preprocess, 6.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/detect/val5

Thermal Domain Validation Results:
mAP50: 0.9714
mAP50-95: 0.7206
Precision: 0.9440
Recall: 0.8974
Images: 526
Instances: 526

Validating on Visible Dataset...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 130.2±69.5 MB/s, size: 413.3 KB)


val: Scanning /content/drive/My Drive/pguard/val/labels.cache... 638 images, 0 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 80/80 [00:06<00:00, 12.18it/s]


                   all        638        956      0.945      0.875      0.944       0.77
Speed: 0.2ms preprocess, 5.6ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/val6

Visible Domain Validation Results:
mAP50: 0.9439
mAP50-95: 0.7699
Precision: 0.9447
Recall: 0.8750
Images: 638
Instances: 956


In [ ]:
from ultralytics import YOLO
import os
import numpy as np

# Define paths
model_path = '/content/drive/MyDrive/pguard/domain_adaptation_process1/visible_to_thermal_semisupervised/weights/best.pt'
thermal_yaml = '/content/Domain-adaptation-object-detection-with-YOLOv8/data/thermal.yaml'
visible_yaml = '/content/Domain-adaptation-object-detection-with-YOLOv8/data/visible.yaml'

# Verify that files exist
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found at {model_path}")
if not os.path.exists(thermal_yaml):
    raise FileNotFoundError(f"Thermal YAML file not found at {thermal_yaml}")
if not os.path.exists(visible_yaml):
    raise FileNotFoundError(f"Visible YAML file not found at {visible_yaml}")

# Load the model
model = YOLO(model_path)

# Function to print validation results
def print_results(results, domain_name, num_images, num_instances):
    print(f"\n{domain_name} Domain Validation Results:")
    print(f"mAP50: {results.box.map50:.4f}")
    print(f"mAP50-95: {results.box.map:.4f}")
    # Handle Precision and Recall as arrays or scalars
    precision = results.box.p[0] if isinstance(results.box.p, (list, np.ndarray)) else results.box.p
    recall = results.box.r[0] if isinstance(results.box.r, (list, np.ndarray)) else results.box.r
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Images: {num_images}")
    print(f"Instances: {num_instances}")

# Validate on thermal dataset
print("\nValidating on Thermal Dataset...")
try:
    thermal_results = model.val(data=thermal_yaml, batch=8, device='0')  # Use GPU (CUDA:0)

except Exception as e:
    print(f"Error during thermal validation: {e}")

# Validate on visible dataset
print("\nValidating on Visible Dataset...")
try:
    visible_results = model.val(data=visible_yaml, batch=8, device='0')  # Use GPU (CUDA:0)

except Exception as e:
    print(f"Error during visible validation: {e}")


Validating on Thermal Dataset...
Ultralytics 8.3.131 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 33.3MB/s]


val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 1.1±0.1 MB/s, size: 439.4 KB)


val: Scanning /content/drive/My Drive/pguard/thermal/val/labels.cache... 526 images, 0 backgrounds, 0 corrupt: 100%|██████████| 526/526 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 66/66 [00:07<00:00,  8.67it/s]


                   all        526        526      0.944      0.897      0.971      0.721
Speed: 0.3ms preprocess, 6.8ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs/detect/val

Validating on Visible Dataset...
Ultralytics 8.3.131 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.6±0.5 ms, read: 0.9±0.2 MB/s, size: 376.1 KB)


val: Scanning /content/drive/My Drive/pguard/val/labels.cache... 638 images, 0 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 80/80 [00:27<00:00,  2.94it/s]


                   all        638        956      0.945      0.875      0.944       0.77
Speed: 0.2ms preprocess, 5.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/val2


In [ ]:
import os

def clear_dataset_caches(dataset_dirs):
    for dir_path in dataset_dirs:
        cache_path = os.path.join(dir_path, 'labels.cache')
        if os.path.exists(cache_path):
            os.remove(cache_path)
            print(f"Deleted cache: {cache_path}")
        else:
            print(f"No cache found: {cache_path}")

dataset_dirs = [
    '/content/drive/My Drive/pguard/thermal/train',
    '/content/drive/My Drive/pguard/thermal/val',
    '/content/drive/My Drive/pguard/train',
    '/content/drive/My Drive/pguard/val',
    '/content/drive/My Drive/pguard/thermal_labeled/train',
    '/content/drive/My Drive/pguard/thermal_labeled/val',
]

clear_dataset_caches(dataset_dirs)

Deleted cache: /content/drive/My Drive/pguard/thermal/train/labels.cache
Deleted cache: /content/drive/My Drive/pguard/thermal/val/labels.cache
Deleted cache: /content/drive/My Drive/pguard/train/labels.cache
Deleted cache: /content/drive/My Drive/pguard/val/labels.cache
No cache found: /content/drive/My Drive/pguard/thermal_labeled/train/labels.cache
No cache found: /content/drive/My Drive/pguard/thermal_labeled/val/labels.cache


In [ ]:
import torch
torch.cuda.empty_cache()
print("CUDA cache cleared.")

CUDA cache cleared.


In [ ]:
!pip install opencv-python pillow

In [ ]:
!pip install numpy==1.25.2 pandas

In [ ]:
logging.getLogger('ultralytics').setLevel(logging.INFO)
LOGGER.setLevel(logging.INFO)

In [ ]:
!cat train.py | grep "def get_dataset"

    def get_dataset(self, data, require_labels=True):


In [ ]:
import os

# Define paths from thermal.yaml
train_dir = "/content/drive/My Drive/pguard/thermal/train/images"
val_dir = "/content/drive/My Drive/pguard/thermal/val/images"

# Function to count images in a directory
def count_images(directory):
    if not os.path.exists(directory):
        return f"Directory {directory} does not exist."
    # Count files with common image extensions
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    image_count = sum(1 for f in os.listdir(directory) if f.lower().endswith(image_extensions))
    return image_count

# Count images in train and validation sets
train_count = count_images(train_dir)
val_count = count_images(val_dir)

# Print results
print(f"Thermal Training Set: {train_count} images")
print(f"Thermal Validation Set: {val_count} images")

Thermal Training Set: 1441 images
Thermal Validation Set: 350 images


In [ ]:
!yolo train resume=True model=runs/detect/visible_to_thermal_semisupervised/weights/last.pt



Ultralytics 8.3.111 🚀 Python-3.11.12 torch-2.0.1+cu117 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=runs/detect/visible_to_thermal_semisupervised/weights/last.pt, data=/content/Domain-adaptation-object-detection-with-YOLOv8/data/visible.yaml, epochs=40, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=None, name=visible_to_thermal_semisupervised, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=9527, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=runs/detect/visible_to_thermal_semisupervised/weights/last.pt, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classe

In [ ]:
# Uninstall NumPy, PyTorch, and Ultralytics
!pip uninstall -y numpy torch torchvision torchaudio ultralytics

# Install compatible versions
!pip install numpy==1.26.4
!pip install torch==2.0.1+cu117 torchvision==0.15.2+cu117 torchaudio==2.0.2+cu117 -f https://download.pytorch.org/whl/torch_stable.html
!pip install ultralytics



Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: torch 2.0.1+cu117
Uninstalling torch-2.0.1+cu117:
  Successfully uninstalled torch-2.0.1+cu117
Found existing installation: torchvision 0.15.2+cu117
Uninstalling torchvision-0.15.2+cu117:
  Successfully uninstalled torchvision-0.15.2+cu117
Found existing installation: torchaudio 2.0.2+cu117
Uninstalling torchaudio-2.0.2+cu117:
  Successfully uninstalled torchaudio-2.0.2+cu117
Found existing installation: ultralytics 8.3.129
Uninstalling ultralytics-8.3.129:
  Successfully uninstalled ultralytics-8.3.129
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following d

In [ ]:
from ultralytics import YOLO

# Load the best model
model = YOLO(' /content/drive/MyDrive/Domain-adaptation-object-detection-with-YOLOv8/runs/detect/visible_to_thermal_semisupervised/weights/best.pt')

# Evaluate on the thermal validation set with num_workers=0
results = model.val(data='/content/drive/My Drive/pguard/thermal/thermal.yaml', split='val', cache=False, workers=0)
print(results)

Ultralytics 8.3.111 🚀 Python-3.11.12 torch-2.0.1+cu117 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 77.9MB/s]


val: Fast image access ✅ (ping: 0.6±0.1 ms, read: 0.9±0.6 MB/s, size: 447.1 KB)


val: Scanning /content/drive/My Drive/pguard/thermal/val/labels.cache... 350 images, 0 backgrounds, 0 corrupt: 100%|██████████| 350/350 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 22/22 [03:54<00:00, 10.66s/it]


                   all        350        350      0.858      0.553      0.709      0.394
Speed: 3.2ms preprocess, 640.4ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs/detect/val2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c440ecff650>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033

In [ ]:
# Script to print Ultralytics YOLO validation metrics
def print_validation_metrics():
    # Define the metrics data (extracted from your document)
    metrics = {
        "Precision": 0.8581171961745893,
        "Recall": 0.5529695915605234,
        "mAP50": 0.7092271712566319,
        "mAP50-95": 0.3942572119421192,
        "Fitness": 0.42575420787357054
    }

    # Print the metrics in a formatted way
    print("Validation Metrics")
    print("=" * 20)
    print(f"Precision: {metrics['Precision']:.3f}")
    print(f"Recall: {metrics['Recall']:.3f}")
    print(f"mAP50: {metrics['mAP50']:.3f}")
    print(f"mAP50-95: {metrics['mAP50-95']:.3f}")
    print(f"Fitness: {metrics['Fitness']:.3f}")

# Run the function
if __name__ == "__main__":
    print_validation_metrics()

Validation Metrics
Precision: 0.858
Recall: 0.553
mAP50: 0.709
mAP50-95: 0.394
Fitness: 0.426


In [ ]:
# Uninstall the current version of NumPy
!pip uninstall -y numpy

# Install a compatible version of NumPy
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
!pip install --upgrade --force-reinstall numpy pandas.

ERROR: Invalid requirement: 'pandas.': Expected end or semicolon (after name and no valid version specifier)
    pandas.
          ^


In [ ]:
!cp -r /content/Domain-adaptation-object-detection-with-YOLOv8 \
      /content/drive/MyDrive/Domain-adaptation-object-detection-with-YOLOv8


In [ ]:
!cp -r /content/pguard/domain_adaptation_process \
      /content/drive/MyDrive/Domain-adaptation-object-detection-with-YOLOv8/DA

In [ ]:
import shutil

# Source model file
src = '/content/Domain-adaptation-object-detection-with-YOLOv8/runs/detect/visible_to_thermal_semisupervised/weights/best.pt'

# Destination folder in Drive (create if it doesn't exist)
dst_dir = '/content/drive/MyDrive/yolov8_models_fordomainadaptation/'
os.makedirs(dst_dir, exist_ok=True)

# Copy
shutil.copy(src, dst_dir)


'/content/drive/MyDrive/yolov8_models_fordomainadaptation/best.pt'